# PL ROI -> JV Assignment  (NOMAD NORTH)

Wire the cell-cropper into NOMAD: pick a **batch** and a **PL image**, overlay the
bundled **5x5 cell mask**, tweak the ROIs, **export** the cropped ROIs, then load the
batch's **group JV files**, split them into **per-device JVs**, and **assign** each ROI
to its device. The **Assign** step writes `manifest.json` - the map of the dataset.

**Products** (overlay PNG, labelled ROI TIFFs, per-device JV txt, `manifest.json`) are
pushed back to the selected batch's NOMAD folder under a date/time-stamped subfolder.

Run the two cells below. Everything else happens in the app UI.

*Bundled next to this notebook:* `cell_layout_5x5.dxf`, `helmholtz_theme.py`,
`nomad_data.py`, `jv_parsing.py`, `api_calls.py`, `utils.py`, `bootstrap.py`.


In [ ]:
# Setup: make sure required packages are present, then load the Helmholtz CSS.
# On NOMAD NORTH most are already installed; missing ones are pip-installed quietly.
from bootstrap import ensure_requirements
ensure_requirements(extra=[("skimage", "scikit-image")])

# NOMAD access token: read from the NOMAD_CLIENT_ACCESS_TOKEN environment variable
# on NORTH. You can also paste a token into the "Token" field in the app and press
# Connect. (api_calls.get_token(url, name) can mint one from username/password.)
import os
print("Token from environment:", "set" if os.environ.get("NOMAD_CLIENT_ACCESS_TOKEN") else "not set - paste one in the app")


In [ ]:
from __future__ import annotations

import base64, io, json, math, zipfile
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np

import matplotlib
matplotlib.use("module://ipympl.backend_nbagg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
plt.close('all')

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

import ipywidgets as widgets
from IPython.display import display

try:
    import tifffile
except Exception as e:
    raise RuntimeError("Missing dependency: tifffile. Install with `pip install tifffile`.") from e

# ── integration: NOMAD client, Helmholtz theme, JV parser ──────────────────────
import os
import datetime as _dt
import helmholtz_theme as hz
import nomad_data as nd
import jv_parsing as jv
import api_calls as _apicalls
import utils as _utils

# The DXF mask ships next to this notebook — no upload needed.
def _find_bundled_dxf():
    candidates = ["cell_layout_5x5.dxf",
                  os.path.join(os.getcwd(), "cell_layout_5x5.dxf")]
    for c in candidates:
        if os.path.exists(c):
            return c
    return "cell_layout_5x5.dxf"   # let the open() error be explicit if truly missing

BUNDLED_DXF_PATH = _find_bundled_dxf()
NOMAD_URL   = nd.URL_API
NOMAD_TOKEN = _utils.TOKEN          # from NOMAD_CLIENT_ACCESS_TOKEN on NOMAD NORTH

# Optional: scikit-image for corner detection (preferred), fallback to pure numpy
try:
    from skimage.feature import corner_harris, corner_peaks, corner_subpix
    from skimage.filters import gaussian
    _SKIMAGE = True
except ImportError:
    _SKIMAGE = False

# ── constants ──────────────────────────────────────────────────────────────────
MASK_SIZE_CM       = 5.0
LABEL_OFFSET_X_PX  = 0
LABEL_OFFSET_Y_PX  = -10
GREEN              = "#90EE90"
YELLOW             = "#FFF59D"
SNAP_RING_COLOR    = "#FF3333"   # red ring shown in magnifier and on main image near snap point
MAGNIFIER_HALF_WIN = 40          # half-size of the magnifier crop in image pixels
SCOPE_HALF_WIN     = 7           # corner search restricted to this radius (image px) around cursor

# Corner detection parameters (used in find_best_corner_in_crop)
HARRIS_K          = 0.05
HARRIS_SIGMA      = 2.0

# Canny sensitivity presets: (blur_sigma, low_threshold, high_threshold)
CANNY_PRESETS = {
    "Normal":    (1.2, 0.05, 0.15),
    "High":      (0.8, 0.02, 0.08),
    "Very High": (0.4, 0.01, 0.04),
}


# ── data classes ───────────────────────────────────────────────────────────────
@dataclass
class Polyline:
    pts: np.ndarray

@dataclass
class ROI:
    label: str
    poly_cm: np.ndarray
    bbox_cm: Tuple[float,float,float,float]
    centroid_cm: Tuple[float,float]


# ── generic helpers ────────────────────────────────────────────────────────────
def _bytes_from_upload(content):
    if content is None:                return None
    if isinstance(content, str):       return content.encode("utf-8", errors="ignore")
    if isinstance(content, memoryview):return content.tobytes()
    if isinstance(content, bytearray): return bytes(content)
    if isinstance(content, bytes):     return content
    return bytes(content)

def _upload_value_first(uploader: widgets.FileUpload):
    val = uploader.value
    if not val: return None
    if isinstance(val, dict):          return next(iter(val.values()))
    if isinstance(val, (list, tuple)): return val[0]
    return val

def sort_corners_tltrblbr(pts: list) -> np.ndarray:
    """Sort 4 (x,y) clicks into [TL, TR, BL, BR] by coordinate geometry."""
    arr        = np.array(pts, dtype=float)
    idx        = np.argsort(arr[:, 1])
    top_two    = arr[idx[:2]];  top_two    = top_two[np.argsort(top_two[:,0])]
    bottom_two = arr[idx[2:]];  bottom_two = bottom_two[np.argsort(bottom_two[:,0])]
    return np.vstack([top_two, bottom_two])          # TL, TR, BL, BR

def _safe_range(arr: np.ndarray) -> float:
    return float(np.max(arr)) - float(np.min(arr))


# ── DXF parser ─────────────────────────────────────────────────────────────────
def parse_dxf_lwpolylines(dxf_bytes: bytes) -> List[Polyline]:
    try:    text = dxf_bytes.decode("utf-8",  errors="ignore")
    except: text = dxf_bytes.decode("latin-1",errors="ignore")
    lines = [ln.rstrip("\r\n") for ln in text.splitlines()]
    i = 0; polys: List[Polyline] = []
    in_entities = False; in_lwpoly = False; pts: List[List[float]] = []

    def _finalize():
        nonlocal pts, in_lwpoly
        if not in_lwpoly: return
        arr = np.array(pts, dtype=float)
        arr = arr[~np.isnan(arr).any(axis=1)]
        if len(arr) >= 3: polys.append(Polyline(arr))
        pts = []; in_lwpoly = False

    while i < len(lines) - 1:
        code = lines[i].strip(); val = lines[i+1].strip(); i += 2
        if code == "2" and val == "ENTITIES": in_entities = True;  continue
        if code == "0" and val == "ENDSEC":   _finalize(); in_entities = False; continue
        if not in_entities: continue
        if code == "0":
            _finalize()
            if val == "LWPOLYLINE": in_lwpoly = True; pts = []
            continue
        if in_lwpoly:
            if code == "10": pts.append([float(val), np.nan])
            elif code == "20":
                y = float(val)
                if pts and math.isnan(pts[-1][1]): pts[-1][1] = y
                else: pts.append([np.nan, y])
    _finalize()
    return polys

def normalize_polys_to_mask_cm(polys: List[Polyline]) -> List[np.ndarray]:
    if not polys: return []
    all_pts = np.concatenate([p.pts for p in polys], axis=0)
    xmin,ymin = np.min(all_pts[:,0]), np.min(all_pts[:,1])
    xmax,ymax = np.max(all_pts[:,0]), np.max(all_pts[:,1])
    w = xmax-xmin; h = ymax-ymin
    if w<=0 or h<=0: return []
    out = []
    for p in polys:
        pts = p.pts.copy()
        pts[:,0] = (pts[:,0]-xmin)/w*MASK_SIZE_CM
        pts[:,1] = (pts[:,1]-ymin)/h*MASK_SIZE_CM
        out.append(pts)
    return out

def poly_bbox_centroid(pts: np.ndarray):
    xs=pts[:,0]; ys=pts[:,1]
    return (float(xs.min()),float(ys.min()),float(xs.max()),float(ys.max())), \
           (float(xs.mean()),float(ys.mean()))

def infer_rois(polys_cm: List[np.ndarray]) -> List[ROI]:
    items = []
    for pts in polys_cm:
        bbox,cent = poly_bbox_centroid(pts)
        area = (bbox[2]-bbox[0])*(bbox[3]-bbox[1])
        items.append((area,pts,bbox,cent))
    items.sort(key=lambda t: t[0])
    if len(items)>24: items=items[:24]
    mid = MASK_SIZE_CM/2.0
    buckets: Dict[int,List] = {1:[],2:[],3:[],4:[]}
    for _,pts,bbox,(cx,cy) in items:
        left=cx<mid; bottom=cy<mid
        # Quadrant numbering (as seen by user):
        #   top-left=2, top-right=1, bottom-left=4, bottom-right=3
        # Note: in the code's coordinates `bottom` (cy<mid) corresponds to the
        # visually-top half, so the mapping below looks "inverted" vs. the names.
        if   left  and     bottom: qid=2      # top-left  (visually)
        elif (not left) and bottom: qid=1      # top-right
        elif left  and (not bottom): qid=4     # bottom-left
        else: qid=3                            # bottom-right
        buckets[qid].append((pts,bbox,(cx,cy)))
    rois: List[ROI] = []
    for qid in [1,2,3,4]:
        group   = buckets[qid]
        group.sort(key=lambda t:(t[2][1],t[2][0]))
        # Left-side quadrants (now 2 and 4) use reversed letter order so that
        # 'f' lands at the outer-left corner and 'a' nearest the image centre.
        letters = list("fedcba") if qid in (2,4) else list("abcdef")
        for idx,(pts,bbox,cent) in enumerate(group[:6]):
            letter = letters[idx] if idx<6 else f"x{idx}"
            rois.append(ROI(f"{qid}-{letter}",pts,bbox,cent))
    return rois


# ── homography ─────────────────────────────────────────────────────────────────
def homography_from_4pts(src: np.ndarray, dst: np.ndarray) -> np.ndarray:
    A=[]
    for (x,y),(u,v) in zip(src,dst):
        A.append([x,y,1,0,0,0,-u*x,-u*y,-u])
        A.append([0,0,0,x,y,1,-v*x,-v*y,-v])
    A=np.array(A,float); _,_,Vt=np.linalg.svd(A)
    h=Vt[-1,:]; H=h.reshape(3,3)
    return H/H[2,2]

def apply_homography(pts: np.ndarray, H: np.ndarray) -> np.ndarray:
    p=np.c_[pts,np.ones((pts.shape[0],1))]
    q=(H@p.T).T
    return q[:,:2]/q[:,2:3]


# ── polygon mask ───────────────────────────────────────────────────────────────
def poly_mask_in_bbox(poly_px: np.ndarray, xmin:int, ymin:int,
                       xmax:int, ymax:int) -> np.ndarray:
    from matplotlib.path import Path
    H=ymax-ymin+1; W=xmax-xmin+1
    ys,xs=np.mgrid[0:H,0:W]
    pts_grid=np.c_[xs.ravel()+xmin, ys.ravel()+ymin]
    mask=Path(poly_px).contains_points(pts_grid).reshape(H,W)
    return mask

def _get_roi_pixels(roi: ROI, img16: np.ndarray, H: np.ndarray):
    Himg,Wimg=img16.shape[:2]
    poly_px=apply_homography(roi.poly_cm,H)
    xs=poly_px[:,0]; ys=poly_px[:,1]
    xmin=int(max(0,math.floor(xs.min())))
    xmax=int(min(Wimg-1,math.ceil(xs.max())))
    ymin=int(max(0,math.floor(ys.min())))
    ymax=int(min(Himg-1,math.ceil(ys.max())))
    mask=poly_mask_in_bbox(poly_px,xmin,ymin,xmax,ymax)
    crop=img16[ymin:ymax+1,xmin:xmax+1].copy()
    crop[~mask]=0
    return crop,mask,poly_px


# ── ruler ticks ────────────────────────────────────────────────────────────────
def _draw_ruler_ticks(ax, H_cm_to_px):
    for cm in range(0,6):
        p  =apply_homography(np.array([[float(cm),5.0 ]],float),H_cm_to_px)[0]
        pe =apply_homography(np.array([[float(cm),5.15]],float),H_cm_to_px)[0]
        ax.plot([p[0],pe[0]],[p[1],pe[1]],color=GREEN,linewidth=1.5)
        pl =apply_homography(np.array([[float(cm),5.25]],float),H_cm_to_px)[0]
        ax.text(pl[0],pl[1],f"{cm}",color=GREEN,fontsize=9,ha="center",va="bottom")
        if cm<5:
            for mm in range(1,10):
                cv=cm+mm*0.1
                pm =apply_homography(np.array([[cv,5.0 ]],float),H_cm_to_px)[0]
                pme=apply_homography(np.array([[cv,5.08]],float),H_cm_to_px)[0]
                ax.plot([pm[0],pme[0]],[pm[1],pme[1]],color=GREEN,linewidth=0.8)
    for cm in range(0,6):
        y_cm=5.0-float(cm)
        q  =apply_homography(np.array([[0.0, y_cm]],float),H_cm_to_px)[0]
        qe =apply_homography(np.array([[0.15,y_cm]],float),H_cm_to_px)[0]
        ax.plot([q[0],qe[0]],[q[1],qe[1]],color=GREEN,linewidth=1.5)
        ql =apply_homography(np.array([[0.25,y_cm]],float),H_cm_to_px)[0]
        ax.text(ql[0],ql[1],f"{cm}",color=GREEN,fontsize=9,ha="left",va="center")
        if cm<5:
            for mm in range(1,10):
                cv=cm+mm*0.1; y_cm_m=5.0-cv
                qm =apply_homography(np.array([[0.0, y_cm_m]],float),H_cm_to_px)[0]
                qme=apply_homography(np.array([[0.08,y_cm_m]],float),H_cm_to_px)[0]
                ax.plot([qm[0],qme[0]],[qm[1],qme[1]],color=GREEN,linewidth=0.8)


# ══════════════════════════════════════════════════════════════════════════════
# ── AUTOMATIC CORNER DETECTION ────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def _to_float32_normalized(img16: np.ndarray) -> np.ndarray:
    """Convert uint16 image to float32 in [0,1] with percentile stretch."""
    lo = float(np.percentile(img16, 1))
    hi = float(np.percentile(img16, 99.5))
    if hi <= lo: hi = lo + 1.0
    return np.clip((img16.astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)



def snap_to_candidate(click_xy: Tuple[float,float],
                       candidates: np.ndarray,
                       radius: float = 50.0) -> Tuple[float,float]:
    """
    If a corner candidate is within `radius` pixels of click_xy, return
    the candidate position instead.  Otherwise return click_xy unchanged.
    """
    if candidates is None or len(candidates) == 0:
        return click_xy
    cx, cy = click_xy
    dists = np.linalg.norm(candidates - np.array([[cx, cy]]), axis=1)
    best  = int(np.argmin(dists))
    if dists[best] <= radius:
        return (float(candidates[best, 0]), float(candidates[best, 1]))
    return click_xy


def _line_from_two_points(p1, p2):
    """Return (a,b,c) s.t. ax+by+c=0 through p1,p2."""
    x1,y1 = p1; x2,y2 = p2
    a = y2-y1; b = x1-x2; c = x2*y1 - x1*y2
    return (a, b, c)

def _intersect_lines(L1, L2):
    """Intersect two lines given as (a,b,c). Returns (x,y) or None."""
    a1,b1,c1 = L1; a2,b2,c2 = L2
    det = a1*b2 - a2*b1
    if abs(det) < 1e-9: return None
    x = (b1*c2 - b2*c1) / det
    y = (a2*c1 - a1*c2) / det
    return (x, y)

def find_best_corner_in_crop(img16: np.ndarray,
                              cx_px: float, cy_px: float,
                              half_win: int):
    """
    Find the sharpest corner within the magnifier crop by:
      1. Detect edges (Canny)
      2. Fit the two most prominent straight-edge orientations (via gradient angles)
      3. Intersect them → precise corner point

    Falls back to Harris if edge intersection fails.
    Returns (x, y) in full-image pixel coords, plus
            (edge1_pts, edge2_pts) for drawing — both in full-image coords,
            or (None, None) if unavailable.
    """
    Himg, Wimg = img16.shape[:2]
    y0 = max(0, int(round(cy_px)) - half_win)
    y1 = min(Himg, int(round(cy_px)) + half_win)
    x0 = max(0, int(round(cx_px)) - half_win)
    x1 = min(Wimg, int(round(cx_px)) + half_win)
    crop = img16[y0:y1, x0:x1]
    H, W = crop.shape[:2]
    if H < 8 or W < 8:
        return None, None, None

    # Normalise to float32 [0,1] with local contrast stretch
    lo = float(np.percentile(crop, 1)); hi = float(np.percentile(crop, 99))
    if hi <= lo: return None, None, None
    fc = np.clip((crop.astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
    fc8 = (fc * 255).astype(np.uint8)

    # ── Step 1: Canny edge map ────────────────────────────────────────────
    c_sigma, c_low, c_high = CANNY_PRESETS[radio_sensitivity.value]
    try:
        if _SKIMAGE:
            from skimage.feature import canny as sk_canny
            from skimage.filters import gaussian as sk_gauss
            blurred = sk_gauss(fc, sigma=c_sigma)
            edges   = sk_canny(blurred, sigma=0.5, low_threshold=c_low, high_threshold=c_high)
        else:
            import cv2
            blurred = cv2.GaussianBlur(fc8, (5,5), c_sigma)
            edges   = cv2.Canny(blurred, int(c_low*255), int(c_high*255)).astype(bool)
    except Exception:
        edges = None

    corner_crop = None   # (col, row) in crop coords
    line1_crop  = None   # two (col,row) endpoints for drawing
    line2_crop  = None

    if edges is not None and edges.sum() > 10:
        # ── Step 2: Gradient directions on edge pixels ────────────────────
        gy, gx = np.gradient(fc)
        edge_rows, edge_cols = np.where(edges)
        if len(edge_rows) >= 6:
            angles = np.arctan2(gy[edge_rows, edge_cols],
                                gx[edge_rows, edge_cols])  # -π..π
            # Edge normal direction → edge runs perpendicular
            # Bin angles into [0°,180°) (edges are undirected)
            angles_deg = (np.degrees(angles) % 180).astype(int)

            # Find the two dominant angle clusters (each edge has one dominant direction)
            hist, bins = np.histogram(angles_deg, bins=36, range=(0, 180))
            # Peak 1
            p1 = int(np.argmax(hist))
            # Suppress neighbourhood of peak 1, find peak 2
            hist2 = hist.copy()
            for d in range(-4, 5):
                hist2[(p1 + d) % 36] = 0
            p2 = int(np.argmax(hist2))

            def _pts_in_bin(bin_idx, tol=2):
                lo_a = bin_idx * 5; hi_a = lo_a + 5
                mask = ((angles_deg >= max(0, lo_a - tol*5)) &
                        (angles_deg <  min(180, hi_a + tol*5)))
                idxs = np.where(mask)[0]
                if len(idxs) < 2: return None
                cols_e = edge_cols[idxs].astype(float)
                rows_e = edge_rows[idxs].astype(float)
                return cols_e, rows_e

            def _fit_line(cols_e, rows_e):
                """Fit a line to edge points; return (a,b,c) and two endpoints."""
                if len(cols_e) < 2: return None, None
                # PCA-based line fit
                cx_ = cols_e.mean(); cy_ = rows_e.mean()
                dc = cols_e - cx_; dr = rows_e - cy_
                cov = np.array([[np.dot(dc,dc), np.dot(dc,dr)],
                                [np.dot(dc,dr), np.dot(dr,dr)]]) / len(cols_e)
                vals, vecs = np.linalg.eigh(cov)
                # Principal direction = eigenvector with largest eigenvalue
                dx, dy = vecs[:, 1]   # direction along line
                # Normal: (-dy, dx)  → line eq: -dy*(x-cx_) + dx*(y-cy_) = 0
                a = -dy; b = dx; c = dy*cx_ - dx*cy_
                # Endpoints for drawing: project range of pts onto direction
                t  = dc*dx + dr*dy
                t1 = float(t.min()); t2 = float(t.max())
                ep1 = (cx_ + t1*dx, cy_ + t1*dy)
                ep2 = (cx_ + t2*dx, cy_ + t2*dy)
                return (a, b, c), (ep1, ep2)

            res1 = _pts_in_bin(p1)
            res2 = _pts_in_bin(p2)
            if res1 is not None and res2 is not None:
                L1, ends1 = _fit_line(*res1)
                L2, ends2 = _fit_line(*res2)
                if L1 is not None and L2 is not None:
                    pt = _intersect_lines(L1, L2)
                    # Accept only if inside (or very near) the crop
                    margin = half_win * 0.5
                    if pt is not None:
                        px_c, py_c = pt
                        if (-margin <= px_c <= W + margin and
                            -margin <= py_c <= H + margin):
                            corner_crop = (px_c, py_c)
                            line1_crop  = ends1
                            line2_crop  = ends2

    # ── Fallback: Harris within crop ──────────────────────────────────────
    if corner_crop is None:
        if _SKIMAGE:
            smooth   = gaussian(fc, sigma=1.5)
            response = corner_harris(smooth, k=HARRIS_K)
            peaks    = corner_peaks(response, min_distance=3, num_peaks=1,
                                    threshold_rel=0.05)
            if len(peaks) > 0:
                try:
                    sub = corner_subpix(smooth, peaks, window_size=7)
                    bad = np.any(np.isnan(sub), axis=1)
                    sub[bad] = peaks[bad].astype(float)
                except Exception:
                    sub = peaks.astype(float)
                corner_crop = (float(sub[0,1]), float(sub[0,0]))  # col, row
        else:
            from scipy.ndimage import gaussian_filter, maximum_filter
            smooth = gaussian_filter(fc, sigma=1.5)
            Iy, Ix = np.gradient(smooth)
            Ixx = gaussian_filter(Ix*Ix, sigma=1.5)
            Iyy = gaussian_filter(Iy*Iy, sigma=1.5)
            Ixy = gaussian_filter(Ix*Iy, sigma=1.5)
            det = Ixx*Iyy - Ixy**2; tr = Ixx + Iyy
            R = det - HARRIS_K * tr**2
            if R.max() > 0:
                lm  = maximum_filter(R, size=5)
                pts = np.argwhere((R == lm) & (R > R.max()*0.05))
                if len(pts) > 0:
                    bi = int(np.argmax(R[pts[:,0], pts[:,1]]))
                    corner_crop = (float(pts[bi,1]), float(pts[bi,0]))

    if corner_crop is None:
        return None, None, None

    # Convert crop coords → full-image coords
    cx_full = x0 + corner_crop[0]
    cy_full = y0 + corner_crop[1]

    def _to_full(pts_crop):
        if pts_crop is None: return None
        return ((x0 + pts_crop[0][0], y0 + pts_crop[0][1]),
                (x0 + pts_crop[1][0], y0 + pts_crop[1][1]))

    return (cx_full, cy_full), _to_full(line1_crop), _to_full(line2_crop)


# ── app state ──────────────────────────────────────────────────────────────────
state = {
    "tiff_uint16":         None,
    "tiff_filename":       "",
    "tiff_preview":        None,
    "mask_rois":           [],
    "mode":                "top",
    "sam_corners_px":      [],
    "H_cm_to_px":          None,
    "H_px_to_cm":          None,
    # ── new corner-detection state ──
    "corner_candidates":   None,   # np.ndarray (N,2) or None — used only for Auto-pick 4
    "auto_corners_shown":  False,
    "snap_enabled":        True,
    "local_snap_pt":       None,   # (x,y) or None
    "local_edge1":         None,   # two (x,y) full-image endpoints or None
    "local_edge2":         None,
    # ── ROI tweak state ──
    "hovered_roi":         None,
    "active_roi":          None,
    "active_roi_backup":   None,
    "selected_rois":       set(),          # multi-select: set of labels
    "selected_backups":    {},             # label → (poly_cm_copy, centroid_cm) for each selected
    "reference_roi":       None,           # the ROI whose position others align to
    "roi_offsets":         {},
    "roi_originals":       {},   # label → (poly_cm_copy, centroid_cm, bbox_cm) at load time
    "undo_stack":          [],   # list of snapshots (each: dict label → (poly_cm, centroid_cm, bbox_cm))
    "redo_stack":          [],   # list of snapshots
    # ── NOMAD + JV-assignment integration state ──
    "nomad_url":           NOMAD_URL,
    "nomad_token":         NOMAD_TOKEN,
    "batch_id":            None,
    "samples":             [],
    "image_files":         [],     # list of dicts from nd.list_image_files
    "selected_image":      None,   # dict for the loaded mother image
    "image_sample_id":     None,
    "jv_files":            [],     # raw group-JV files from nd.list_jv_files
    "per_device_jv":       {},     # device_id -> {"bytes","source_file"}
    "jv_metrics":          {},     # device_id -> {Jsc_rev,...}
    "exported_rois":       {},     # roi_label -> {"tiff": bytes, "record": {...}}
    "overlay_png":         None,   # bytes of mask-overlayed mother image
    "roi_records":         [],     # geometry records (same as old export csv rows)
    "assignment":          {},     # (legacy) roi_label -> {device_id, jv_source_file, jv_bytes, metrics}
    "bucket":              {},     # roi_label -> {device_id, jv_source_file, jv_bytes, metrics, roi_tiff, roi_record, quadrant}
    "bucket_sealed":       False,
}


# ── widgets ────────────────────────────────────────────────────────────────────
# ── NOMAD source widgets (replace the old TIFF/DXF file-upload inputs) ──────────
dd_batch           = widgets.Combobox(options=[], description="Batch:",
                                      placeholder="type to filter…", ensure_option=True,
                                      layout=widgets.Layout(width="360px"))
btn_refresh_batches = widgets.Button(description="↻", tooltip="Reload batch list from NOMAD",
                                     layout=widgets.Layout(width="40px"))
btn_load_batch     = widgets.Button(description="Load batch", button_style="primary",
                                    tooltip="List this batch's images in the Image dropdown")
chk_only_tiff      = widgets.Checkbox(value=True, description="only tiff images",
                                      indent=False, layout=widgets.Layout(width="160px"))
source_out         = widgets.Output()   # batch/image diagnostics (never fails silently)
dd_image           = widgets.Dropdown(options=[], description="Image:",
                                      layout=widgets.Layout(width="420px"))
btn_load_image     = widgets.Button(description="Load image", button_style="primary",
                                    tooltip="Load the selected PL image into the preview")
# The DXF mask is bundled; this button (re)loads it from disk if ever needed.
btn_load_mask      = widgets.Button(description="Reload bundled mask",
                                    layout=widgets.Layout(width="170px"))
radio_mode        = widgets.RadioButtons(options=[("Top view","top"),("SAM","sam")],
                                         value="top",description="Mode:")
btn_reset_corners = widgets.Button(description="Reset corners")
btn_reset_rois    = widgets.Button(description="↺ Reset ROI positions", button_style="warning",
                                   tooltip="Restore all ROIs to their original positions",
                                   layout=widgets.Layout(width="190px"))
btn_overlay       = widgets.Button(description="Overlay mask", button_style="success")
btn_export        = widgets.Button(description="Export ROI",   button_style="warning")
status            = widgets.HTML("<b>Status:</b> Ready")
export_out        = widgets.HTML("")

# ── NEW: auto-corner widgets ──────────────────────────────────────────────────
toggle_snap = widgets.ToggleButton(
    value=True,
    description="🧲 Snap ON",
    button_style="info",
    tooltip="When ON, clicks snap to the detected corner inside the scope",
    layout=widgets.Layout(width="120px"),
)
radio_scope = widgets.RadioButtons(
    options=[("Large", SCOPE_HALF_WIN),
             ("Medium", SCOPE_HALF_WIN // 2),
             ("Small",  max(2, SCOPE_HALF_WIN // 3))],
    value=SCOPE_HALF_WIN,
    description="Scope:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="200px"),
)
radio_sensitivity = widgets.RadioButtons(
    options=["Normal", "High", "Very High"],
    value="Normal",
    description="Sensitivity:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="200px"),
)
radio_overlay_zoom = widgets.RadioButtons(
    options=[("Close (60px)", 60), ("Medium (120px)", 120), ("Wide (200px)", 200)],
    value=60,
    description="Overlay mag view:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="240px"),
)
corner_info = widgets.HTML("")

def set_status(msg, color="black"):
    status.value = f"<b>Status:</b> <span style='color:{color}'>{msg}</span>"


# ── main figures ───────────────────────────────────────────────────────────────
plt.close('all')

fig_img,     ax_img     = plt.subplots(figsize=(5,5))
fig_mag,     ax_mag     = plt.subplots(figsize=(5,5))
fig_mask,    ax_mask    = plt.subplots(figsize=(5,5))
fig_overlay, ax_overlay = plt.subplots(figsize=(5,5))
fig_omag,    ax_omag    = plt.subplots(figsize=(5,5))   # overlay magnifier

ax_img.set_title("Preview 1: TIFF (click corners in SAM)"); ax_img.axis("off")
ax_mag.set_title("Magnifier");                               ax_mag.axis("off")
ax_mask.set_title("Preview 2: Mask")
ax_mask.set_aspect("equal","box")
ax_mask.set_xlim(0,MASK_SIZE_CM); ax_mask.set_ylim(0,MASK_SIZE_CM)
ax_mask.invert_yaxis(); ax_mask.axis("off")
ax_overlay.set_title("Preview 3: Overlay");                  ax_overlay.axis("off")
ax_omag.set_title("Overlay Magnifier");                      ax_omag.axis("off")


# ── draw helpers ───────────────────────────────────────────────────────────────
def _draw_hover_snap_on_ax(ax, hover_x=None, hover_y=None):
    """
    Draw a subtle snap-target ring on the main image ONLY when the cursor is
    near a candidate.  No permanent clutter — the ring follows the mouse.
    """
    cands = state["corner_candidates"]
    if cands is None or len(cands) == 0: return
    if hover_x is None or hover_y is None: return
    if not state["snap_enabled"]: return

    snapped = snap_to_candidate((hover_x, hover_y), cands, SCOPE_HALF_WIN * 3)
    if snapped == (hover_x, hover_y): return   # cursor not near any candidate

    sx, sy = snapped
    # Outer thin white ring (contrast halo)
    ax.add_patch(Circle((sx, sy), radius=9, color="white",
                         fill=False, linewidth=2.5, alpha=0.6, zorder=8))
    # Inner red ring — matches what appears in magnifier
    ax.add_patch(Circle((sx, sy), radius=7, color=SNAP_RING_COLOR,
                         fill=False, linewidth=2.0, alpha=0.9, zorder=9))


def draw_image_preview(hover_x=None, hover_y=None):
    ax_img.clear(); ax_img.set_title("Preview 1: TIFF"); ax_img.axis("off")
    if state["tiff_preview"] is not None:
        ax_img.imshow(state["tiff_preview"], cmap="gray", origin="upper", interpolation="nearest")

    n_clicks = len(state["sam_corners_px"])
    if state["mode"]=="top" and n_clicks==2:
        pts = sort_corners_tltrblbr(state["sam_corners_px"])
        tl = pts[0]; tr = pts[1]
        side = float(np.linalg.norm(tr - tl))
        dx = tr[0]-tl[0]; dy = tr[1]-tl[1]
        length = math.sqrt(dx*dx+dy*dy) or 1.0
        perp = np.array([-dy/length, dx/length]) * side
        bl = tl + perp; br = tr + perp
        quad = np.array([tl, tr, br, bl, tl])
        ax_img.plot(quad[:,0], quad[:,1], color=YELLOW, linewidth=2)
    elif state["mode"]=="sam" and n_clicks==4:
        sp = sort_corners_tltrblbr(state["sam_corners_px"])
        quad = np.array([sp[0],sp[1],sp[3],sp[2],sp[0]])
        ax_img.plot(quad[:,0], quad[:,1], color=YELLOW, linewidth=2)

    # Yellow crosshairs for already-selected corners
    for (x, y) in state["sam_corners_px"]:
        ax_img.plot([x],[y], marker="+", markersize=10, color=YELLOW,
                    linestyle="None", zorder=7)

    # Clean snap-target ring — only when hovering near a candidate
    _draw_hover_snap_on_ax(ax_img, hover_x, hover_y)

    fig_img.canvas.draw_idle()

def draw_mask_preview():
    ax_mask.clear(); ax_mask.set_title("Preview 2: Mask")
    ax_mask.set_aspect("equal","box"); ax_mask.set_xlim(0,MASK_SIZE_CM)
    ax_mask.set_ylim(0,MASK_SIZE_CM); ax_mask.invert_yaxis(); ax_mask.axis("off")
    rois=state["mask_rois"]
    if not rois: fig_mask.canvas.draw_idle(); return
    ax_mask.plot([0,MASK_SIZE_CM,MASK_SIZE_CM,0,0],[0,0,MASK_SIZE_CM,MASK_SIZE_CM,0],
                 color=GREEN,linewidth=2)
    for roi in rois:
        pts=roi.poly_cm
        ax_mask.plot(np.r_[pts[:,0],pts[0,0]],np.r_[pts[:,1],pts[0,1]],color=GREEN,linewidth=1)
        cx,cy=roi.centroid_cm
        ax_mask.plot([cx],[cy],marker="+",color=GREEN,markersize=8,linestyle="None")
        xmin,ymin,xmax,ymax=roi.bbox_cm
        ax_mask.text(xmax+0.02,ymax-0.02,roi.label,color=GREEN,fontsize=8,ha="left",va="top")
    fig_mask.canvas.draw_idle()

def draw_overlay_preview():
    active  = state["active_roi"]
    hovered = state["hovered_roi"]
    selected = state["selected_rois"]
    title = "Preview 3: Overlay"
    n_undo = len(state["undo_stack"]); n_redo = len(state["redo_stack"])
    undo_info = f"Ctrl+Z=undo({n_undo})  Ctrl+Y=redo({n_redo})"
    if selected:
        n_sel = len(selected)
        ref = state["reference_roi"] or "?"
        title = (f"Preview 3: Overlay  |  {n_sel} sel (ref: {ref})\n"
                 f"R=ref  H=horiz  V=vert  ↑↓←→=move  Enter=confirm  Esc=cancel  {undo_info}")
    elif active:
        title = (f"Preview 3: Overlay  |  ✦ ROI {active} ACTIVE\n"
                 f"↑↓←→=move  R=ref  Ctrl+click=multisel  Enter=confirm  Esc=cancel  {undo_info}")
    else:
        title = f"Preview 3: Overlay\nClick ROI to select  |  {undo_info}"
    ax_overlay.clear(); ax_overlay.set_title(title, fontsize=8 if active else 9); ax_overlay.axis("off")
    if state["tiff_preview"] is None: fig_overlay.canvas.draw_idle(); return
    ax_overlay.imshow(state["tiff_preview"],cmap="gray",origin="upper",interpolation="nearest")
    # Zoom out: add padding around the image
    if state["tiff_preview"] is not None:
        Himg, Wimg = state["tiff_preview"].shape[:2]
        pad_x = Wimg * OVERLAY_ZOOM_OUT_PAD
        pad_y = Himg * OVERLAY_ZOOM_OUT_PAD
        ax_overlay.set_xlim(-pad_x, Wimg + pad_x)
        ax_overlay.set_ylim(Himg + pad_y, -pad_y)  # inverted y for image coords
    rois=state["mask_rois"]; H=state["H_cm_to_px"]
    if (not rois) or H is None: fig_overlay.canvas.draw_idle(); return
    box_cm=np.array([[0,0],[MASK_SIZE_CM,0],[MASK_SIZE_CM,MASK_SIZE_CM],[0,MASK_SIZE_CM]],float)
    box_px=apply_homography(box_cm,H)
    ax_overlay.plot(*np.vstack([box_px,box_px[0]]).T,color=GREEN,linewidth=2)
    _draw_ruler_ticks(ax_overlay,H)

    selected = state["selected_rois"]
    ref_roi  = state["reference_roi"]
    for roi in rois:
        if roi.label == active:
            color = "#2196F3"   # blue — single active/moving
            lw = 2.0
        elif roi.label == ref_roi and (selected or ref_roi is not None):
            color = "#FF9800"   # orange — reference ROI
            lw = 2.5
        elif roi.label in selected:
            color = "#AB47BC"   # purple — multi-selected
            lw = 2.0
        elif roi.label == hovered:
            color = "#FF3333"   # red — hover highlight
            lw = 2.0
        else:
            color = GREEN
            lw = 1.0

        is_highlighted = roi.label in (hovered, active) or roi.label in selected or roi.label == ref_roi
        pts_px = apply_homography(roi.poly_cm, H)
        ax_overlay.plot(*np.vstack([pts_px,pts_px[0]]).T, color=color, linewidth=lw)
        c_px = apply_homography(np.array([roi.centroid_cm],float), H)[0]
        ax_overlay.plot([c_px[0]], [c_px[1]], marker="+", color=color,
                        markersize=10 if is_highlighted else 8,
                        linestyle="None", markeredgewidth=2.0 if is_highlighted else 1.0)
        xs=pts_px[:,0]; ys=pts_px[:,1]
        # Show role tag for reference / selected
        extra = ""
        if roi.label == ref_roi and (selected or ref_roi is not None): extra = " ★ref"
        elif roi.label in selected: extra = " ●sel"
        ax_overlay.text(float((xs.min()+xs.max())/2)+LABEL_OFFSET_X_PX+80,
                        float(ys.min())+LABEL_OFFSET_Y_PX+30,
                        roi.label + extra, color=color, fontsize=9, ha="center", va="bottom",
                        bbox=dict(boxstyle='round,pad=0.3',facecolor='black',edgecolor='none',alpha=0.7))
    fig_overlay.canvas.draw_idle()


OMAG_HALF_WIN = 60    # overlay magnifier crop half-size in image pixels (larger = more context)
OMAG_ZOOM     = 3     # lower zoom than corner magnifier (6×) — just enough to see ROI edges clearly

# ── Overlay preview zoom-out ───────────────────────────────────────────────────
# Fraction of image size added as padding on each side (0.0 = tight fit, 0.2 = 20% margin).
# Increase this value to zoom out further. Tweak here:
OVERLAY_ZOOM_OUT_PAD = 0.15   # ← change this to zoom out (e.g. 0.3) or in (e.g. 0.0)

def _update_overlay_magnifier(x_px: float, y_px: float):
    """Update the overlay magnifier figure — shows TIFF crop + live ROI overlays."""
    preview = state["tiff_preview"]
    if preview is None:
        ax_omag.clear(); ax_omag.set_title("Overlay Magnifier"); ax_omag.axis("off")
        fig_omag.canvas.draw_idle(); return

    H      = state["H_cm_to_px"]
    rois   = state["mask_rois"]
    Himg, Wimg = preview.shape[:2]
    hw = radio_overlay_zoom.value
    z  = OMAG_ZOOM

    # Crop bounds (may be smaller near edges)
    y0 = max(0, int(round(y_px)) - hw); y1 = min(Himg, int(round(y_px)) + hw)
    x0 = max(0, int(round(x_px)) - hw); x1 = min(Wimg, int(round(x_px)) + hw)
    crop = preview[y0:y1, x0:x1]
    if crop.size == 0: return

    zoomed = np.repeat(np.repeat(crop, z, axis=0), z, axis=1)

    ax_omag.clear()
    ax_omag.set_title("Overlay Magnifier", fontsize=9)
    ax_omag.axis("off")
    ch = (x_px - x0) * z   # cursor position in zoomed coords
    cv = (y_px - y0) * z
    ax_omag.imshow(zoomed, cmap="gray", origin="upper",
                   extent=[0, zoomed.shape[1], zoomed.shape[0], 0])

    # Thin crosshair at cursor
    ax_omag.axhline(cv, color="yellow", linewidth=0.7, alpha=0.6)
    ax_omag.axvline(ch, color="yellow", linewidth=0.7, alpha=0.6)

    # Draw ROI polygons + centroids if overlay is active
    if H is not None and rois:
        hovered = state["hovered_roi"]
        active  = state["active_roi"]
        for roi in rois:
            if roi.label == active:
                color = "#2196F3"; lw = 2.0
            elif roi.label == hovered:
                color = "#FF3333"; lw = 2.0
            else:
                color = GREEN;     lw = 1.0

            # Convert poly cm → full px → zoomed crop coords
            pts_full = apply_homography(roi.poly_cm, H)
            pts_z = np.column_stack([(pts_full[:,0] - x0) * z,
                                     (pts_full[:,1] - y0) * z])
            closed = np.vstack([pts_z, pts_z[0]])
            ax_omag.plot(closed[:,0], closed[:,1], color=color,
                         linewidth=lw, alpha=0.9)

            # Centroid
            c_full = apply_homography(np.array([roi.centroid_cm], float), H)[0]
            cx_z = (c_full[0] - x0) * z
            cy_z = (c_full[1] - y0) * z
            ax_omag.plot([cx_z], [cy_z], marker="+", color=color,
                         markersize=10, linestyle="None",
                         markeredgewidth=2.0 if roi.label in (hovered, active) else 1.5)

            # Label if near cursor
            dist_px = math.sqrt((c_full[0]-x_px)**2 + (c_full[1]-y_px)**2)
            if dist_px < hw * 1.5:
                ax_omag.text(cx_z + 4, cy_z - 4, roi.label,
                             color=color, fontsize=7,
                             bbox=dict(boxstyle='round,pad=0.2', facecolor='black',
                                       edgecolor='none', alpha=0.65))

    ax_omag.set_xlim(0, zoomed.shape[1])
    ax_omag.set_ylim(zoomed.shape[0], 0)
    fig_omag.canvas.draw_idle()


# ── magnifier ──────────────────────────────────────────────────────────────────
def _update_magnifier(x_px: float, y_px: float, source_img: np.ndarray):
    if source_img is None: return
    Himg, Wimg = source_img.shape[:2]; hw = MAGNIFIER_HALF_WIN

    y0 = max(0, int(round(y_px)) - hw); y1 = min(Himg, int(round(y_px)) + hw)
    x0 = max(0, int(round(x_px)) - hw); x1 = min(Wimg, int(round(x_px)) + hw)
    crop = source_img[y0:y1, x0:x1]
    if crop.size == 0: return

    z = 6  # fixed magnifier zoom
    zoomed = np.repeat(np.repeat(crop, z, axis=0), z, axis=1)

    ax_mag.clear(); ax_mag.set_title("Magnifier", fontsize=9); ax_mag.axis("off")
    ax_mag.imshow(zoomed, cmap="gray" if zoomed.ndim==2 else None,
                  origin="upper", interpolation="nearest")

    # Yellow cursor crosshair
    ch = (x_px - x0) * z;  cv = (y_px - y0) * z
    ax_mag.axhline(cv, color=YELLOW, linewidth=1.0, alpha=0.65)
    ax_mag.axvline(ch, color=YELLOW, linewidth=1.0, alpha=0.65)

    # Scope circle — matches the search radius exactly
    scope_hw = radio_scope.value
    scope_r_zoomed = scope_hw * z
    ax_mag.add_patch(Circle((ch, cv), scope_r_zoomed,
                             color=YELLOW, fill=False, linewidth=1.5, alpha=0.75))

    # ── Corner detection: only search inside the scope circle ────────────
    scope_hw = radio_scope.value
    corner_pt = None
    if state["snap_enabled"] and state["tiff_uint16"] is not None:
        pt, edge1, edge2 = find_best_corner_in_crop(
            state["tiff_uint16"], x_px, y_px, scope_hw)

        # Only accept if the found corner is actually within the scope radius
        if pt is not None:
            dist_from_cursor = math.sqrt((pt[0]-x_px)**2 + (pt[1]-y_px)**2)
            if dist_from_cursor <= scope_hw:
                corner_pt = pt
            else:
                pt = edge1 = edge2 = None   # outside scope — reject

        state["local_snap_pt"] = corner_pt
        state["local_edge1"]   = edge1 if corner_pt is not None else None
        state["local_edge2"]   = edge2 if corner_pt is not None else None

        if corner_pt is not None:
            sx, sy = corner_pt
            mx = (sx - x0) * z
            my = (sy - y0) * z

            # Green edge lines
            for edge in (state["local_edge1"], state["local_edge2"]):
                if edge is not None:
                    ax_mag.plot([(edge[0][0]-x0)*z, (edge[1][0]-x0)*z],
                                [(edge[0][1]-y0)*z, (edge[1][1]-y0)*z],
                                color="#44FF44", linewidth=2.0,
                                alpha=0.8, zorder=8, solid_capstyle="round")

            # Red circle + crosshair at intersection
            ring_r = max(7, z * 1.8)
            ax_mag.add_patch(Circle((mx, my), radius=ring_r + 2,
                                    color="white", fill=False,
                                    linewidth=2.5, alpha=0.5, zorder=9))
            ax_mag.add_patch(Circle((mx, my), radius=ring_r,
                                    color=SNAP_RING_COLOR, fill=False,
                                    linewidth=2.5, alpha=1.0, zorder=10))
            cs = ring_r * 0.55
            ax_mag.plot([mx-cs, mx+cs], [my,    my   ],
                        color=SNAP_RING_COLOR, linewidth=1.8, zorder=10)
            ax_mag.plot([mx,    mx   ], [my-cs, my+cs],
                        color=SNAP_RING_COLOR, linewidth=1.8, zorder=10)
    else:
        state["local_snap_pt"] = None
        state["local_edge1"]   = None
        state["local_edge2"]   = None

    fig_mag.canvas.draw_idle()

_last_mouse = {"x": None, "y": None}

def _on_motion_tiff(e):
    if e.inaxes == ax_img and e.xdata is not None:
        _last_mouse["x"] = e.xdata; _last_mouse["y"] = e.ydata
        _update_magnifier(e.xdata, e.ydata, state["tiff_preview"])
        _refresh_snap_ring_on_main()

def _on_motion_overlay(e):
    if e.inaxes == ax_overlay and e.xdata is not None:
        _last_mouse["x"] = e.xdata; _last_mouse["y"] = e.ydata
        _update_overlay_magnifier(e.xdata, e.ydata)
        # ── ROI hover detection ──
        if state["H_px_to_cm"] is not None and state["mask_rois"]:
            _update_hovered_roi(e.xdata, e.ydata)

def _roi_contains_px(roi, x_px, y_px, H_px_to_cm):
    """Return True if pixel (x_px,y_px) is inside roi's polygon."""
    from matplotlib.path import Path
    pt_cm = apply_homography(np.array([[x_px, y_px]], float), H_px_to_cm)[0]
    return Path(roi.poly_cm).contains_point(pt_cm)

def _update_hovered_roi(x_px, y_px):
    """Find which ROI (if any) is under the cursor; redraw if changed."""
    if state["active_roi"] is not None:
        return   # don't change hover while a ROI is active
    # Allow hover even during multi-select so user can see what they'll Ctrl+click
    H_px_to_cm = state["H_px_to_cm"]
    prev = state["hovered_roi"]
    found = None
    for roi in state["mask_rois"]:
        if _roi_contains_px(roi, x_px, y_px, H_px_to_cm):
            found = roi.label; break
    if found != prev:
        state["hovered_roi"] = found
        draw_overlay_preview()


# ── ROI tweak: click to activate / confirm ────────────────────────────────────
def _find_roi_under_cursor(x_px, y_px):
    """Return the label of the ROI under (x_px, y_px), or None."""
    H_px_to_cm = state["H_px_to_cm"]
    if H_px_to_cm is None: return None
    for roi in state["mask_rois"]:
        if _roi_contains_px(roi, x_px, y_px, H_px_to_cm):
            return roi.label
    return None

def _on_click_overlay(event):
    if event.inaxes != ax_overlay or event.xdata is None: return
    H = state["H_cm_to_px"]
    if H is None or not state["mask_rois"]: return

    # Right-click → cancel active ROI (and clear multi-select)
    if event.button == 3:
        if state["active_roi"] is not None:
            _cancel_active_roi()
        if state["selected_rois"]:
            _cancel_all_selected()
        return

    if event.button != 1: return

    # Detect Ctrl held (ipympl sends guiEvent with state or key modifier)
    ctrl_held = False
    try:
        if hasattr(event, "guiEvent") and event.guiEvent is not None:
            ge = event.guiEvent
            if hasattr(ge, "ctrlKey"):
                ctrl_held = bool(ge.ctrlKey)
            elif isinstance(ge, dict):
                ctrl_held = bool(ge.get("ctrlKey", False))
        if not ctrl_held and hasattr(event, "key") and event.key is not None:
            ctrl_held = "control" in str(event.key)
    except Exception:
        pass

    active = state["active_roi"]
    selected = state["selected_rois"]

    # ── If there is a single active ROI (old-style active, pre-confirm) ──
    if active is not None and not ctrl_held:
        _confirm_active_roi()
        return

    # ── Ctrl+click: multi-select toggle ──
    if ctrl_held:
        clicked_label = _find_roi_under_cursor(event.xdata, event.ydata)
        if clicked_label is None: return

        # If a single active ROI exists, confirm it first so it becomes the reference
        if active is not None:
            _confirm_active_roi()

        if clicked_label in selected:
            # Deselect
            selected.discard(clicked_label)
            if clicked_label in state["selected_backups"]:
                del state["selected_backups"][clicked_label]
            if state["reference_roi"] == clicked_label:
                state["reference_roi"] = next(iter(selected)) if selected else None
            _update_select_status()
            draw_overlay_preview()
        else:
            # Add to selection — save backup
            roi = next(r for r in state["mask_rois"] if r.label == clicked_label)
            selected.add(clicked_label)
            state["selected_backups"][clicked_label] = (roi.poly_cm.copy(), roi.centroid_cm)
            # First selected becomes reference if none set
            if state["reference_roi"] is None:
                state["reference_roi"] = clicked_label
            _update_select_status()
            draw_overlay_preview()
        return

    # ── Plain click: single ROI activation (original behavior) ──
    if selected:
        # If clicking outside selected ROIs, confirm all and clear
        clicked_label = _find_roi_under_cursor(event.xdata, event.ydata)
        if clicked_label not in selected:
            _confirm_all_selected()
            # Then activate the clicked one if any
            if clicked_label is not None:
                _activate_roi(clicked_label)
            return
        else:
            # Clicking on one of the selected — just acknowledge, use R to set reference
            _update_select_status()
            draw_overlay_preview()
            return

    # No active, no selection — check what's under cursor
    clicked_label = state["hovered_roi"]
    if clicked_label is None:
        clicked_label = _find_roi_under_cursor(event.xdata, event.ydata)

    if clicked_label is not None:
        _activate_roi(clicked_label)

fig_overlay.canvas.mpl_connect("button_press_event", _on_click_overlay)


def _activate_roi(label):
    roi = next(r for r in state["mask_rois"] if r.label == label)
    # Save backup
    state["active_roi"]        = label
    state["active_roi_backup"] = (roi.poly_cm.copy(), roi.centroid_cm)
    state["hovered_roi"]       = None
    # Also add to selected set so keys work seamlessly
    state["selected_rois"].add(label)
    state["selected_backups"][label] = (roi.poly_cm.copy(), roi.centroid_cm)
    set_status(f"ROI {label} active — ↑↓←→ move, R=set as reference. Ctrl+click to add more, H/V=align. Enter=confirm.", "blue")
    draw_overlay_preview()

def _confirm_active_roi():
    label = state["active_roi"]
    state["active_roi"]        = None
    state["active_roi_backup"] = None
    state["hovered_roi"]       = None
    # Keep label in selected_rois — it stays part of the group
    set_status(f"ROI {label} position confirmed. R=set as reference, Ctrl+click more, H/V=align.", "green")
    draw_overlay_preview()

def _cancel_active_roi():
    label  = state["active_roi"]
    backup = state["active_roi_backup"]
    if backup is not None and label is not None:
        roi = next(r for r in state["mask_rois"] if r.label == label)
        roi.poly_cm     = backup[0]
        roi.centroid_cm = backup[1]
        xs = roi.poly_cm[:,0]; ys = roi.poly_cm[:,1]
        roi.bbox_cm = (float(xs.min()), float(ys.min()),
                       float(xs.max()), float(ys.max()))
    state["active_roi"]        = None
    state["active_roi_backup"] = None
    state["hovered_roi"]       = None
    # Also remove from selected
    state["selected_rois"].discard(label)
    state["selected_backups"].pop(label, None)
    if state["reference_roi"] == label:
        state["reference_roi"] = next(iter(state["selected_rois"])) if state["selected_rois"] else None
    set_status(f"ROI {label} move cancelled.", "orange")
    draw_overlay_preview()

# ── Multi-select helpers ──────────────────────────────────────────────────────
def _update_select_status():
    selected = state["selected_rois"]
    ref = state["reference_roi"]
    if not selected:
        set_status("Selection cleared.", "black")
        return
    labels_str = ", ".join(sorted(selected))
    ref_str = f" (ref: {ref})" if ref else ""
    set_status(f"Selected: {labels_str}{ref_str} — R=set ref, H=align horiz, V=align vert. "
               f"↑↓←→ moves all. Enter confirms. Esc cancels.", "#6A1B9A")

def _confirm_all_selected():
    labels = sorted(state["selected_rois"])
    state["selected_rois"]    = set()
    state["selected_backups"] = {}
    state["reference_roi"]    = None
    state["hovered_roi"]      = None
    if labels:
        set_status(f"Confirmed positions for: {', '.join(labels)}.", "green")
    draw_overlay_preview()

def _cancel_all_selected():
    # Restore all selected ROIs to their backups
    for label, backup in state["selected_backups"].items():
        roi = next((r for r in state["mask_rois"] if r.label == label), None)
        if roi is not None and backup is not None:
            roi.poly_cm     = backup[0]
            roi.centroid_cm = backup[1]
            xs = roi.poly_cm[:,0]; ys = roi.poly_cm[:,1]
            roi.bbox_cm = (float(xs.min()), float(ys.min()),
                           float(xs.max()), float(ys.max()))
    n = len(state["selected_rois"])
    state["selected_rois"]    = set()
    state["selected_backups"] = {}
    state["reference_roi"]    = None
    state["hovered_roi"]      = None
    if n:
        set_status(f"Cancelled — {n} ROI(s) restored to previous positions.", "orange")
    draw_overlay_preview()

def _align_selected_horizontal():
    """Align all non-reference selected ROIs\' X centroid to the reference ROI\' X centroid."""
    ref_label = state["reference_roi"]
    selected  = state["selected_rois"]
    if ref_label is None:
        set_status("No reference set. Select a ROI and press R first.", "red"); return
    if len(selected) < 2:
        set_status("Select at least one more ROI (Ctrl+click) to align.", "red"); return
    ref_roi = next((r for r in state["mask_rois"] if r.label == ref_label), None)
    if ref_roi is None: return
    ref_cx = ref_roi.centroid_cm[0]

    # Check if anything will actually move
    will_move = any(
        abs(ref_cx - next((r for r in state["mask_rois"] if r.label == l), None).centroid_cm[0]) > 1e-9
        for l in selected if l != ref_label and next((r for r in state["mask_rois"] if r.label == l), None) is not None
    )
    if will_move:
        _push_undo()

    moved = []
    for label in selected:
        if label == ref_label: continue
        roi = next((r for r in state["mask_rois"] if r.label == label), None)
        if roi is None: continue
        dx = ref_cx - roi.centroid_cm[0]
        if abs(dx) < 1e-9: continue
        _move_roi_by(label, dx, 0.0)
        moved.append(label)

    if moved:
        set_status(f"H-aligned {', '.join(sorted(moved))} to reference {ref_label}.", "#1565C0")
    else:
        set_status("Already aligned horizontally.", "#1565C0")
    draw_overlay_preview()
    if _last_mouse["x"] is not None:
        _update_overlay_magnifier(_last_mouse["x"], _last_mouse["y"])

def _align_selected_vertical():
    """Align all non-reference selected ROIs\' Y centroid to the reference ROI\' Y centroid."""
    ref_label = state["reference_roi"]
    selected  = state["selected_rois"]
    if ref_label is None:
        set_status("No reference set. Select a ROI and press R first.", "red"); return
    if len(selected) < 2:
        set_status("Select at least one more ROI (Ctrl+click) to align.", "red"); return
    ref_roi = next((r for r in state["mask_rois"] if r.label == ref_label), None)
    if ref_roi is None: return
    ref_cy = ref_roi.centroid_cm[1]

    # Check if anything will actually move
    will_move = any(
        abs(ref_cy - next((r for r in state["mask_rois"] if r.label == l), None).centroid_cm[1]) > 1e-9
        for l in selected if l != ref_label and next((r for r in state["mask_rois"] if r.label == l), None) is not None
    )
    if will_move:
        _push_undo()

    moved = []
    for label in selected:
        if label == ref_label: continue
        roi = next((r for r in state["mask_rois"] if r.label == label), None)
        if roi is None: continue
        dy = ref_cy - roi.centroid_cm[1]
        if abs(dy) < 1e-9: continue
        _move_roi_by(label, 0.0, dy)
        moved.append(label)

    if moved:
        set_status(f"V-aligned {', '.join(sorted(moved))} to reference {ref_label}.", "#1565C0")
    else:
        set_status("Already aligned vertically.", "#1565C0")
    draw_overlay_preview()
    if _last_mouse["x"] is not None:
        _update_overlay_magnifier(_last_mouse["x"], _last_mouse["y"])


# ── Arrow-key movement ────────────────────────────────────────────────────────
ROI_STEP_CM = 0.002   # ~0.002 cm per keypress — very fine step

# ── Undo / Redo system ─────────────────────────────────────────────────────────
_UNDO_MAX = 200   # max undo steps

def _snapshot_all_rois():
    """Return a snapshot dict of all ROI positions."""
    snap = {}
    for roi in state["mask_rois"]:
        snap[roi.label] = (roi.poly_cm.copy(), roi.centroid_cm, roi.bbox_cm)
    return snap

def _restore_snapshot(snap):
    """Restore all ROI positions from a snapshot dict."""
    for roi in state["mask_rois"]:
        if roi.label in snap:
            poly, cent, bbox = snap[roi.label]
            roi.poly_cm     = poly.copy()
            roi.centroid_cm = cent
            roi.bbox_cm     = bbox

def _push_undo():
    """Save current state to undo stack and clear redo stack."""
    state["undo_stack"].append(_snapshot_all_rois())
    if len(state["undo_stack"]) > _UNDO_MAX:
        state["undo_stack"] = state["undo_stack"][-_UNDO_MAX:]
    state["redo_stack"].clear()

def _do_undo():
    if not state["undo_stack"]:
        set_status("Nothing to undo.", "gray"); return
    # Save current to redo before restoring
    state["redo_stack"].append(_snapshot_all_rois())
    snap = state["undo_stack"].pop()
    _restore_snapshot(snap)
    n_undo = len(state["undo_stack"])
    n_redo = len(state["redo_stack"])
    set_status(f"Undo ({n_undo} left). Ctrl+Y to redo ({n_redo} available).", "#6A1B9A")
    draw_overlay_preview()
    if _last_mouse["x"] is not None:
        _update_overlay_magnifier(_last_mouse["x"], _last_mouse["y"])

def _do_redo():
    if not state["redo_stack"]:
        set_status("Nothing to redo.", "gray"); return
    # Save current to undo before restoring
    state["undo_stack"].append(_snapshot_all_rois())
    snap = state["redo_stack"].pop()
    _restore_snapshot(snap)
    n_undo = len(state["undo_stack"])
    n_redo = len(state["redo_stack"])
    set_status(f"Redo ({n_redo} left). Ctrl+Z to undo ({n_undo} available).", "#6A1B9A")
    draw_overlay_preview()
    if _last_mouse["x"] is not None:
        _update_overlay_magnifier(_last_mouse["x"], _last_mouse["y"])


def _move_roi_by(label, dx_cm, dy_cm):
    """Shift a single ROI by (dx_cm, dy_cm) in mask coordinates."""
    roi = next((r for r in state["mask_rois"] if r.label == label), None)
    if roi is None: return
    roi.poly_cm = roi.poly_cm + np.array([dx_cm, dy_cm])
    cx, cy = roi.centroid_cm
    roi.centroid_cm = (cx + dx_cm, cy + dy_cm)
    xs = roi.poly_cm[:,0]; ys = roi.poly_cm[:,1]
    roi.bbox_cm = (float(xs.min()), float(ys.min()),
                   float(xs.max()), float(ys.max()))

def _on_key_overlay(event):
    active   = state["active_roi"]
    selected = state["selected_rois"]
    key = event.key

    # ── Ctrl+Z / Ctrl+Y: undo / redo — always available ──
    if key in ("ctrl+z", "ctrl+Z"):
        _do_undo(); return
    if key in ("ctrl+y", "ctrl+Y"):
        _do_redo(); return

    # ── R key: set reference — works whether single active or multi-selected ──
    if key in ("r", "R"):
        # Determine which ROI to make the reference
        target = active  # if a single ROI is active, use that
        if target is None:
            # Try hovered or cursor
            target = state["hovered_roi"]
            if target is None and _last_mouse["x"] is not None:
                target = _find_roi_under_cursor(_last_mouse["x"], _last_mouse["y"])
        if target is not None:
            state["reference_roi"] = target
            # Make sure it is in the selected set
            if target not in selected:
                roi = next((r for r in state["mask_rois"] if r.label == target), None)
                if roi is not None:
                    selected.add(target)
                    state["selected_backups"][target] = (roi.poly_cm.copy(), roi.centroid_cm)
            set_status(f"ROI {target} set as reference ★. Ctrl+click others, then H=align horiz, V=align vert.", "#E65100")
            draw_overlay_preview()
        else:
            set_status("Hover over or click a ROI first, then press R.", "red")
        return

    # ── H key: horizontal alignment to reference ──
    if key in ("h", "H"):
        _align_selected_horizontal(); return

    # ── V key: vertical alignment to reference ──
    if key in ("v", "V"):
        _align_selected_vertical(); return

    # ── Escape / Enter ──
    if key == "escape":
        if active is not None:
            _cancel_active_roi()
        if selected:
            _cancel_all_selected()
        return
    if key == "enter":
        if active is not None:
            _confirm_active_roi()
        if selected:
            _confirm_all_selected()
        return

    # ── Arrow keys: move active ROI or all selected ──
    dx_cm = dy_cm = 0.0
    if   key == "left":  dx_cm = -ROI_STEP_CM
    elif key == "right": dx_cm =  ROI_STEP_CM
    elif key == "up":    dy_cm = -ROI_STEP_CM
    elif key == "down":  dy_cm =  ROI_STEP_CM
    else: return

    # If nothing is active or selected, ignore
    if active is None and not selected: return

    # Push undo before moving
    _push_undo()

    # Move: if there is a single active ROI, move just that; if multi-selected, move all
    if active is not None and len(selected) <= 1:
        _move_roi_by(active, dx_cm, dy_cm)
    else:
        for label in selected:
            _move_roi_by(label, dx_cm, dy_cm)
    draw_overlay_preview()
    if _last_mouse["x"] is not None:
        _update_overlay_magnifier(_last_mouse["x"], _last_mouse["y"])

fig_overlay.canvas.mpl_connect("key_press_event", _on_key_overlay)

# Lightweight ring on main image — uses cached local_snap_pt, no full redraw
_snap_ring_artists = []

def _refresh_snap_ring_on_main():
    global _snap_ring_artists
    for art in _snap_ring_artists:
        try: art.remove()
        except Exception: pass
    _snap_ring_artists = []

    snap_pt = state["local_snap_pt"]
    if snap_pt is None or not state["snap_enabled"]: return

    sx, sy = snap_pt
    halo = Circle((sx, sy), radius=9, color="white",
                  fill=False, linewidth=2.5, alpha=0.5, zorder=8)
    ring = Circle((sx, sy), radius=7, color=SNAP_RING_COLOR,
                  fill=False, linewidth=2.0, alpha=0.95, zorder=9)
    ax_img.add_patch(halo)
    ax_img.add_patch(ring)
    _snap_ring_artists = [halo, ring]
    fig_img.canvas.draw_idle()

fig_img.canvas.mpl_connect("motion_notify_event", _on_motion_tiff)
fig_overlay.canvas.mpl_connect("motion_notify_event", _on_motion_overlay)

def _on_click_overlay_focus(event):
    """Ensure the overlay canvas captures key events in ipympl/Voila."""
    try: fig_overlay.canvas._key_press_handler_id  # just touch it
    except Exception: pass

fig_overlay.canvas.mpl_connect("button_press_event", _on_click_overlay_focus)
fig_overlay.canvas.capture_scroll = True   # helps focus in some ipympl builds


# ── SAM corner clicks ──────────────────────────────────────────────────────────
def on_click_image(event):
    if event.inaxes!=ax_img or event.xdata is None: return
    max_clicks = 2 if state["mode"]=="top" else 4
    if len(state["sam_corners_px"]) >= max_clicks: return

    raw_xy = (float(event.xdata), float(event.ydata))

    # Use the local snap point computed by the last magnifier update
    snap_pt = state["local_snap_pt"]
    if state["snap_enabled"] and snap_pt is not None:
        final_xy = snap_pt
        dx = final_xy[0]-raw_xy[0]; dy = final_xy[1]-raw_xy[1]
        dist = math.sqrt(dx*dx+dy*dy)
        set_status(f"Snapped to edge intersection (offset {dist:.1f} px)", "blue")
    else:
        final_xy = raw_xy

    state["sam_corners_px"].append(final_xy)
    n = len(state["sam_corners_px"])

    if state["mode"]=="top":
        msg = ("2 points selected — press Overlay to apply square region."
               if n==2 else
               "Click a second point (TR) to define the square, or press Overlay for full image.")
        set_status(msg, "green" if n==2 else "black")
    else:
        set_status(f"{'4 corners — press Overlay.' if n==4 else f'Corner {n}/4 selected.'}",
                   "green" if n==4 else "black")
    draw_image_preview()

fig_img.canvas.mpl_connect("button_press_event", on_click_image)

def handle_mode_change(change):
    state["mode"] = change["new"]
    if state["mode"]=="sam":
        set_status("SAM: click any 4 corners (order doesn't matter). "
                   "Detected candidates shown in cyan — clicks snap to nearest.")
    else:
        set_status("Top view: press Overlay for full image, OR click TL+TR to define a square region.")
    draw_image_preview()

radio_mode.observe(handle_mode_change, names="value")


# ── load / overlay / reset ─────────────────────────────────────────────────────
def _apply_tiff_bytes(raw_bytes, fname):
    """Load TIFF bytes into the preview. Same imaging logic as before; only the
    byte source changed (was a FileUpload, now a NOMAD raw file)."""
    state["tiff_filename"] = os.path.splitext(fname)[0] if fname else ""
    img = np.asarray(tifffile.imread(io.BytesIO(raw_bytes)))
    # Colour (RGB/RGBA) frames -> luminance. Previously a colour image was
    # collapsed by indexing its first axis, which produced the twisted /
    # duplicated-pixel look on angled-view substrates. Handle it explicitly.
    if img.ndim >= 3 and img.shape[-1] in (3, 4):
        rgb = img[..., :3].astype(np.float32)
        img = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
    # Multi-page stacks (N, H, W) -> first plane.
    while img.ndim > 2:
        img = img[0]
    img = np.ascontiguousarray(img)
    if img.dtype != np.uint16: img = img.astype(np.uint16)
    state["tiff_uint16"] = img
    lo = np.percentile(img,1); hi = np.percentile(img,99.5)
    if hi<=lo: hi = lo+1
    state["tiff_preview"] = np.clip((img.astype(np.float32)-lo)/(hi-lo),0,1)
    state["sam_corners_px"] = []
    state["H_cm_to_px"] = None; state["H_px_to_cm"] = None
    state["corner_candidates"] = None
    corner_info.value = ""
    # A freshly loaded image invalidates any previous ROI export / assignment.
    state["exported_rois"] = {}; state["overlay_png"] = None; state["roi_records"] = []
    state["assignment"] = {}
    state["bucket"] = {}; state["bucket_sealed"] = False
    set_status(f"Image loaded: {img.shape[1]}x{img.shape[0]} uint16.", "green")
    draw_image_preview(); draw_overlay_preview()
    # Force a full (not deferred) redraw so a new image never shows a torn frame.
    for _f in (fig_img, fig_overlay):
        try: _f.canvas.draw(); _f.canvas.flush_events()
        except Exception: pass


def load_image_from_nomad(_):
    """Download the image selected in the dropdown from NOMAD and load it."""
    try:
        sel = dd_image.value
        if not sel:
            set_status("No image selected.","red"); return
        set_status("Downloading image from NOMAD…","#005AA0")
        raw = nd.download_raw_file(state["nomad_url"], state["nomad_token"],
                                   sel["upload_id"], sel["path"])
        state["selected_image"]   = sel
        state["image_sample_id"]  = sel["sample_id"]
        _apply_tiff_bytes(raw, sel["name"])
        _refresh_export_lists()
        _reset_bucket_ui()
    except Exception as e:
        set_status(f"Image load error: {e}","red")

btn_load_image.on_click(load_image_from_nomad)

def load_mask(_=None):
    try:
        with open(BUNDLED_DXF_PATH, "rb") as _fh:
            dxf_bytes = _fh.read()
        polys = parse_dxf_lwpolylines(dxf_bytes)
        state["mask_rois"] = infer_rois(normalize_polys_to_mask_cm(polys))
        # Snapshot original positions for reset
        state["roi_originals"] = {
            roi.label: (roi.poly_cm.copy(), roi.centroid_cm, roi.bbox_cm)
            for roi in state["mask_rois"]
        }
        state["active_roi"] = None; state["active_roi_backup"] = None
        state["hovered_roi"] = None; state["roi_offsets"] = {}
        state["undo_stack"] = []; state["redo_stack"] = []
        state["selected_rois"] = set(); state["selected_backups"] = {}; state["reference_roi"] = None
        set_status(f"Bundled mask loaded: {len(polys)} polylines, {len(state['mask_rois'])} ROIs.","green")
        draw_mask_preview(); draw_overlay_preview()
    except Exception as e:
        set_status(f"Mask load error: {e}","red")

btn_load_mask.on_click(load_mask)

def reset_corners(_):
    state["sam_corners_px"] = []
    set_status("Corners reset.")
    draw_image_preview()

btn_reset_corners.on_click(reset_corners)

def reset_roi_positions(_):
    """Restore all ROIs to their original positions from when the mask was loaded."""
    originals = state["roi_originals"]
    if not originals:
        set_status("No original positions saved — load a mask first.", "red"); return
    for roi in state["mask_rois"]:
        if roi.label in originals:
            poly0, cent0, bbox0 = originals[roi.label]
            roi.poly_cm     = poly0.copy()
            roi.centroid_cm = cent0
            roi.bbox_cm     = bbox0
    state["active_roi"]        = None
    state["active_roi_backup"] = None
    state["hovered_roi"]       = None
    state["selected_rois"]     = set()
    state["selected_backups"]  = {}
    state["reference_roi"]     = None
    state["roi_offsets"]       = {}
    state["undo_stack"]        = []
    state["redo_stack"]        = []
    set_status("All ROI positions reset to original.", "orange")
    draw_overlay_preview()

btn_reset_rois.on_click(reset_roi_positions)

def overlay_mask(_):
    try:
        if state["tiff_preview"] is None:  set_status("Load TIFF first.","red");     return
        if not state["mask_rois"]:          set_status("Load DXF mask first.","red"); return
        Himg,Wimg = state["tiff_preview"].shape[:2]
        src = np.array([[0,0],[MASK_SIZE_CM,0],[0,MASK_SIZE_CM],[MASK_SIZE_CM,MASK_SIZE_CM]],float)
        if state["mode"]=="top":
            n_clicks = len(state["sam_corners_px"])
            if n_clicks == 0:
                dst = np.array([[0,0],[Wimg-1,0],[0,Himg-1],[Wimg-1,Himg-1]],float)
            elif n_clicks == 2:
                pts = sort_corners_tltrblbr(state["sam_corners_px"])
                tl = pts[0]; tr = pts[1]
                side = float(np.linalg.norm(tr - tl))
                dx = tr[0]-tl[0]; dy = tr[1]-tl[1]
                length = math.sqrt(dx*dx+dy*dy) or 1.0
                perp = np.array([-dy/length, dx/length]) * side
                bl = tl + perp; br = tr + perp
                dst = np.array([tl, tr, bl, br], float)
            else:
                set_status("Top view: click 0 points (full image) or 2 points (TL + TR to define square). Reset and try again.","red")
                return
        else:
            if len(state["sam_corners_px"])!=4:
                set_status("SAM: click 4 corners first.","red"); return
            dst = sort_corners_tltrblbr(state["sam_corners_px"])
        H = homography_from_4pts(src, dst)
        state["H_cm_to_px"] = H; state["H_px_to_cm"] = np.linalg.inv(H)
        set_status("Overlay applied.","green"); draw_overlay_preview()
    except Exception as e:
        set_status(f"Overlay error: {e}","red")

btn_overlay.on_click(overlay_mask)


# ══════════════════════════════════════════════════════════════════════════════
# ── AUTO CORNER BUTTON HANDLERS ───────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def on_toggle_snap(change):
    state["snap_enabled"] = change["new"]
    toggle_snap.description = "🧲 Snap ON" if change["new"] else "🧲 Snap OFF"
    toggle_snap.button_style = "info" if change["new"] else ""

toggle_snap.observe(on_toggle_snap, names="value")

def on_scope_change(change):
    if _last_mouse["x"] is not None:
        _update_magnifier(_last_mouse["x"], _last_mouse["y"], state["tiff_preview"])

radio_scope.observe(on_scope_change, names="value")

def on_sensitivity_change(change):
    if _last_mouse["x"] is not None:
        _update_magnifier(_last_mouse["x"], _last_mouse["y"], state["tiff_preview"])

radio_sensitivity.observe(on_sensitivity_change, names="value")

def on_overlay_zoom_change(change):
    x = _last_mouse["x"]
    y = _last_mouse["y"]
    if x is None and state["tiff_preview"] is not None:
        # No mouse position yet — use image centre as fallback
        Himg, Wimg = state["tiff_preview"].shape[:2]
        x, y = Wimg / 2, Himg / 2
    if x is not None:
        _update_overlay_magnifier(x, y)

radio_overlay_zoom.observe(on_overlay_zoom_change, names="value")


# ── ROI crop display helper ────────────────────────────────────────────────────
def _get_roi_display_crop(roi, img16, H):
    """Return a masked ROI crop suitable for display (float, masked array)."""
    Himg, Wimg = img16.shape[:2]
    poly_px = apply_homography(roi.poly_cm, H)
    xs = poly_px[:, 0]; ys = poly_px[:, 1]
    xmin = int(max(0, math.floor(xs.min())))
    xmax = int(min(Wimg - 1, math.ceil(xs.max())))
    ymin = int(max(0, math.floor(ys.min())))
    ymax = int(min(Himg - 1, math.ceil(ys.max())))
    mask = poly_mask_in_bbox(poly_px, xmin, ymin, xmax, ymax)
    crop = img16[ymin:ymax + 1, xmin:xmax + 1].astype(np.float32)
    # Normalize for display
    valid = crop[mask]
    if valid.size > 0:
        lo = float(np.percentile(valid, 1))
        hi = float(np.percentile(valid, 99))
        if hi <= lo: hi = lo + 1.0
        crop = np.clip((crop - lo) / (hi - lo), 0.0, 1.0)
    # Use masked array so background is transparent
    return np.ma.masked_where(~mask, crop)



# ── export ROI crops ───────────────────────────────────────────────────────────
def export_all(_):
    try:
        if state["tiff_uint16"] is None or state["H_cm_to_px"] is None:
            set_status("Load image, mask, and apply overlay first.","red"); return
        img16=state["tiff_uint16"]; H=state["H_cm_to_px"]; rois=state["mask_rois"]

        fig=plt.figure(figsize=(6,6)); ax=fig.add_subplot(111); ax.axis("off")
        ax.imshow(state["tiff_preview"],cmap="gray",origin="upper")
        box_cm=np.array([[0,0],[MASK_SIZE_CM,0],[MASK_SIZE_CM,MASK_SIZE_CM],[0,MASK_SIZE_CM]],float)
        box_px=apply_homography(box_cm,H)
        ax.plot(*np.vstack([box_px,box_px[0]]).T,color=GREEN,linewidth=2)
        _draw_ruler_ticks(ax,H)
        for roi in rois:
            pts_px=apply_homography(roi.poly_cm,H)
            ax.plot(*np.vstack([pts_px,pts_px[0]]).T,color=GREEN,linewidth=1)
            c_px=apply_homography(np.array([roi.centroid_cm],float),H)[0]
            ax.plot([c_px[0]],[c_px[1]],marker="+",color=GREEN,markersize=8,linestyle="None")
            xs=pts_px[:,0]; ys=pts_px[:,1]
            ax.text(float((xs.min()+xs.max())/2)+LABEL_OFFSET_X_PX+80,
                    float(ys.min())+LABEL_OFFSET_Y_PX+30,
                    roi.label,color=GREEN,fontsize=9,ha="center",va="bottom",
                    bbox=dict(boxstyle='round,pad=0.3',facecolor='black',edgecolor='none',alpha=0.7))
        buf=io.BytesIO(); fig.savefig(buf,format="png",dpi=200,bbox_inches="tight",pad_inches=0.02)
        plt.close(fig); overlay_png=buf.getvalue()

        import csv
        records=[]; roi_tiffs={}
        for roi in rois:
            crop,mask,poly_px=_get_roi_pixels(roi,img16,H)
            xs_px=poly_px[:,0]; ys_px=poly_px[:,1]
            xmin=int(max(0,math.floor(xs_px.min())))
            ymin_b=int(max(0,math.floor(ys_px.min())))
            c_px=apply_homography(np.array([roi.centroid_cm],float),H)[0]
            tb=io.BytesIO(); tifffile.imwrite(tb,crop.astype(np.uint16),photometric="minisblack")
            roi_tiffs[f"{roi.label}.tif"]=tb.getvalue()
            records.append({"roi":roi.label,
                             "centroid_x_px":float(c_px[0]),"centroid_y_px":float(c_px[1]),
                             "centroid_x_cm":float(roi.centroid_cm[0]),
                             "centroid_y_cm":float(MASK_SIZE_CM-roi.centroid_cm[1]),
                             "bbox_xmin_px":xmin,"bbox_ymin_px":ymin_b,
                             "bbox_xmax_px":xmin+crop.shape[1]-1,
                             "bbox_ymax_px":ymin_b+crop.shape[0]-1})
        csv_io=io.StringIO()
        w=csv.DictWriter(csv_io,fieldnames=list(records[0].keys()) if records else [])
        w.writeheader()
        for r in records: w.writerow(r)
        csv_bytes=csv_io.getvalue().encode("utf-8")
        json_bytes=json.dumps(records,indent=2).encode("utf-8")

        zip_buf=io.BytesIO()
        with zipfile.ZipFile(zip_buf,"w",compression=zipfile.ZIP_DEFLATED,allowZip64=True) as z:
            z.writestr("overlay.png",overlay_png)
            z.writestr("rois.csv",csv_bytes); z.writestr("rois.json",json_bytes)
            for name,data in roi_tiffs.items(): z.writestr(f"rois/{name}",data)
        zip_buf.seek(0)
        with zipfile.ZipFile(zip_buf,"r") as z:
            bad=z.testzip()
            if bad: raise RuntimeError(f"ZIP corrupted at {bad}")
        b64=base64.b64encode(zip_buf.getvalue()).decode("ascii")
        _prefix = (state["tiff_filename"] + "_") if state["tiff_filename"] else ""
        _zip_name = f"{_prefix}export_rois.zip"
        export_out.value=(f'<a download="{_zip_name}" '
                          f'href="data:application/zip;base64,{b64}">'
                          f'⬇ Download {_zip_name}</a>')
        # ── integration: keep products in state, show thumbnails, refresh lists ──
        state["overlay_png"] = overlay_png
        state["roi_records"] = records
        state["exported_rois"] = {
            r["roi"]: {"tiff": roi_tiffs[f"{r['roi']}.tif"], "record": r}
            for r in records
        }
        state["assignment"] = {}          # a new export invalidates old pairing
        state["bucket"] = {}; state["bucket_sealed"] = False
        _render_export_thumbnails()
        _refresh_export_lists()
        _reset_bucket_ui()
        set_status("Export ready.","green")
    except Exception as e:
        import traceback; traceback.print_exc()
        set_status(f"Export error: {e}","red")

btn_export.on_click(export_all)


# ══════════════════════════════════════════════════════════════════════════════
# ── NOMAD WIRING + JV ASSIGNMENT (integration) ────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

# ── extra widgets for the integration ──────────────────────────────────────────
export_thumbs   = widgets.HTML("")                         # ROI thumbnails after Export
jv_status       = widgets.HTML("<b>JV:</b> no files loaded")
sel_rois        = widgets.SelectMultiple(options=[], description="Exported ROIs:",
                                         rows=10, layout=widgets.Layout(width="320px"),
                                         style={"description_width":"110px"})
sel_jvs         = widgets.SelectMultiple(options=[], description="Per-device JVs:",
                                         rows=10, layout=widgets.Layout(width="360px"),
                                         style={"description_width":"110px"})
# Manual browse of the batch for JV text files (handles files nested in folders).
btn_browse_jv   = widgets.Button(description="Browse batch for JV", button_style="info",
                                 tooltip="List every .txt file in this batch (incl. subfolders) to pick from")
sel_jv_browse   = widgets.SelectMultiple(options=[], description="JV files:", rows=6,
                                         layout=widgets.Layout(width="680px"),
                                         style={"description_width":"70px"})
btn_load_selected_jv = widgets.Button(description="Load selected JV", button_style="info",
                                 tooltip="Download & parse the ticked group JV file(s) into per-device measurements")
jv_chips_box    = widgets.Box(layout=widgets.Layout(display="flex", flex_flow="row wrap",
                                                    align_items="flex-start"))
jv_detail_out   = widgets.Output()   # side panel: JV curve + metrics for the clicked chip
bucket_out      = widgets.Output()   # ROI ↔ JV bucket columns
btn_assign      = widgets.Button(description="Assign (add bundle)", button_style="success",
                                 tooltip="Pair the loaded JVs with matching ROIs by label and add the bundle to the bucket")
btn_seal        = widgets.Button(description="Seal the bucket", button_style="primary",
                                 tooltip="Finalize the bucket and enable export")
btn_export_nomad = widgets.Button(description="Export bucket → NOMAD", button_style="warning",
                                  disabled=True,
                                  tooltip="Upload the sealed bucket: overlay PNG, ROI TIFFs, per-device JVs and the ML-manifest")
assign_out      = widgets.Output()
nomad_out       = widgets.Output()
manifest_dl     = widgets.HTML("")


# ── NOMAD: batches → images ─────────────────────────────────────────────────────
def refresh_batches(_=None):
    try:
        set_status("Loading batches from NOMAD…","#005AA0")
        batches = nd.list_batches(state["nomad_url"], state["nomad_token"])
        dd_batch.options = batches
        set_status(f"{len(batches)} batches loaded.","green")
    except Exception as e:
        set_status(f"Batch load error: {e}","red")

def load_batch(_=None):
    import traceback
    batch_id = (dd_batch.value or "").strip()
    state["batch_id"] = batch_id
    dd_image.options = []
    sel_jvs.options = []
    state["per_device_jv"] = {}; state["jv_metrics"] = {}
    jv_status.value = "<b>JV:</b> no files loaded"
    source_out.clear_output()
    if not batch_id:
        set_status("Type or select a batch first.","red"); return
    if dd_batch.options and batch_id not in dd_batch.options:
        set_status(f"'{batch_id}' is not in the batch list.","red"); return
    try:
        set_status("Loading samples & images…","#005AA0")
        samples = nd.list_samples_in_batch(state["nomad_url"], state["nomad_token"], batch_id)
        state["samples"] = samples
        exts = (".tif", ".tiff") if chk_only_tiff.value else None
        imgs = nd.list_image_files(state["nomad_url"], state["nomad_token"], samples, exts=exts)
        state["image_files"] = imgs
        dd_image.options = [(d["label"], d) for d in imgs]
        kind = "tiff images" if chk_only_tiff.value else "files"
        set_status(f"{len(samples)} sample(s), {len(imgs)} {kind} in {batch_id}.",
                   "green" if imgs else "#B25900")
        # Always show what happened so an empty list is explainable, not silent.
        with source_out:
            print("Batch:", batch_id)
            for line in nd.LAST_DIAGNOSTICS:
                print(" ", line)
            if not imgs:
                print("  (no images matched — untick 'only tiff images' to list every file,")
                print("   or check that this batch's upload actually contains image files)")
    except Exception as exc:
        set_status(f"Batch contents error: {exc}","red")
        with source_out:
            print("ERROR while loading batch:", exc)
            traceback.print_exc()
            for line in nd.LAST_DIAGNOSTICS:
                print(" ", line)

btn_refresh_batches.on_click(refresh_batches)
btn_load_batch.on_click(load_batch)


# ── NOMAD: group JV files → per-device JVs ──────────────────────────────────────


# ── browse the batch for JV files (incl. nested folders) + parse selected ───────
def browse_jv(_=None):
    try:
        sample_ids = state["samples"] or ([state["image_sample_id"]] if state["image_sample_id"] else [])
        if not sample_ids:
            jv_status.value = "<b style='color:#C62828'>JV:</b> load a batch first"; return
        jv_status.value = "<b>JV:</b> browsing…"
        files = nd.browse_files(state["nomad_url"], state["nomad_token"], sample_ids, exts=(".txt",))
        sel_jv_browse.options = [(f["label"], f) for f in files]
        jv_status.value = (f"<b style='color:#2E7D32'>JV:</b> {len(files)} .txt file(s) found — "
                           f"tick the group JV file(s) and press 'Load selected JV'")
    except Exception as e:
        jv_status.value = f"<b style='color:#C62828'>JV browse error:</b> {e}"

def load_selected_jv(_=None):
    try:
        picked = list(sel_jv_browse.value)
        if not picked:
            jv_status.value = "<b style='color:#C62828'>JV:</b> tick at least one file"; return
        jv_status.value = "<b>JV:</b> downloading…"
        pairs = []
        for f in picked:
            raw = nd.download_raw_file(state["nomad_url"], state["nomad_token"],
                                       f["upload_id"], f["path"])
            pairs.append((f["name"], raw))
        state["jv_files"] = picked
        state["per_device_jv"] = jv.split_many(pairs)
        metrics = {}
        for name, raw in pairs:
            metrics.update(jv.parse_jv_header_metrics(raw, name))
        state["jv_metrics"] = metrics
        n_dev = len(state["per_device_jv"])
        jv_status.value = (f"<b style='color:#2E7D32'>JV:</b> {len(picked)} file(s) → "
                           f"{n_dev} per-device measurements")
        _refresh_export_lists()
        _render_jv_chips()
    except Exception as e:
        jv_status.value = f"<b style='color:#C62828'>JV load error:</b> {e}"

btn_browse_jv.on_click(browse_jv)
btn_load_selected_jv.on_click(load_selected_jv)


# ── per-device JV chips with right-click curve+metrics popups ────────────────────
def _parse_device_curve(raw_bytes):
    """Pull (voltage, current_rev, current_fwd) curve points from a per-device
    3-column TSV. Rows whose first column is a metric name (non-numeric) are
    skipped, leaving the JV sweep points."""
    xs, rev, fwd = [], [], []
    for ln in raw_bytes.decode("utf-8", "ignore").splitlines():
        p = ln.split("\t")
        if len(p) < 3:
            continue
        try:
            v = float(p[0]); r = float(p[1]); f = float(p[2])
        except ValueError:
            continue
        xs.append(v); rev.append(r); fwd.append(f)
    return xs, rev, fwd

def _render_jv_chips(*_):
    """One clickable button per per-device JV. Clicking shows that device's full
    JV curve + metrics in the dedicated side panel."""
    per = state["per_device_jv"]
    if not per:
        jv_chips_box.children = [widgets.HTML(
            "<i style='color:#5B6B7B;font-family:Arial'>No JV measurements loaded yet.</i>")]
        jv_detail_out.clear_output()
        return
    btns = []
    for did in sorted(per):
        b = widgets.Button(description=f"\U0001F4C8 {did}",
                           tooltip=f"Show JV curve & metrics for {did}",
                           layout=widgets.Layout(width="84px", margin="2px"))
        b.add_class("jvchip-btn")
        b.on_click(lambda _b, d=did: _show_jv_detail(d))
        btns.append(b)
    jv_chips_box.children = btns

def _show_jv_detail(did):
    """Render one device's JV curve + metrics into the side panel."""
    from IPython.display import HTML, display
    info = state["per_device_jv"].get(did)
    jv_detail_out.clear_output(wait=True)
    with jv_detail_out:
        if not info:
            print("No data for", did); return
        xs, rev, fwd = _parse_device_curve(info["bytes"])
        fig = plt.figure(figsize=(4.3, 3.2)); ax = fig.add_subplot(111)
        if xs:
            ax.plot(xs, rev, "-",  color=hz.HZ_BLUE, lw=1.9, label="reverse")
            ax.plot(xs, fwd, "--", color=hz.HZ_RED,  lw=1.5, label="forward")
            ax.axhline(0, color="#999", lw=.6); ax.axvline(0, color="#999", lw=.6)
            ax.set_xlabel("Voltage (V)"); ax.set_ylabel("Current density")
            ax.set_title(f"JV  {did}", fontsize=11, color=hz.HZ_BLUE_DARK)
            ax.legend(fontsize=9); ax.grid(alpha=.3)
        else:
            ax.text(.5, .5, "no curve points in file", ha="center", va="center"); ax.axis("off")
        bb = io.BytesIO(); fig.savefig(bb, format="png", dpi=92, bbox_inches="tight"); plt.close(fig)
        img64 = base64.b64encode(bb.getvalue()).decode("ascii")
        m = state["jv_metrics"].get(did, {})
        rows = "".join(
            f"<tr><td style='padding:1px 12px'>{k}</td>"
            f"<td style='padding:1px 12px;text-align:right;font-variant-numeric:tabular-nums'>{v:.3f}</td></tr>"
            for k, v in m.items())
        mtable = (f"<table style='border-collapse:collapse;font-size:12px;margin-top:4px'>"
                  f"<tr><th colspan='2' style='color:{hz.HZ_BLUE_DARK};text-align:left'>Metrics</th></tr>"
                  f"{rows}</table>") if m else "<i>no header metrics</i>"
        display(HTML(f"<div style='font-family:Arial'>"
                     f"<div style='font-size:12px;color:{hz.HZ_MUTED}'>source: {info['source_file']}</div>"
                     f"<img src='data:image/png;base64,{img64}' style='max-width:440px;display:block'/>"
                     f"{mtable}</div>"))


# ── list refresh + export thumbnails ────────────────────────────────────────────
def _refresh_export_lists(*_):
    sel_rois.options = [(lbl, lbl) for lbl in sorted(state["exported_rois"])]
    sel_jvs.options = [(f"{did}   ← {info['source_file']}", did)
                       for did, info in sorted(state["per_device_jv"].items())]

def _render_export_thumbnails(*_):
    rois = state["exported_rois"]
    if not rois or state["tiff_uint16"] is None or state["H_cm_to_px"] is None:
        export_thumbs.value = ""; return
    img16 = state["tiff_uint16"]; H = state["H_cm_to_px"]
    by_label = {r.label: r for r in state["mask_rois"]}
    cells = []
    for lbl in sorted(rois):
        roi = by_label.get(lbl)
        if roi is None:
            continue
        disp = _get_roi_display_crop(roi, img16, H)
        fig = plt.figure(figsize=(1.1, 1.1)); ax = fig.add_subplot(111); ax.axis("off")
        ax.imshow(disp, cmap="gray")   # match how the ROIs are actually cropped/previewed
        b = io.BytesIO(); fig.savefig(b, format="png", dpi=70,
                                      bbox_inches="tight", pad_inches=0); plt.close(fig)
        b64 = base64.b64encode(b.getvalue()).decode("ascii")
        cells.append(
            f"<div style='display:inline-block;text-align:center;margin:3px;'>"
            f"<img src='data:image/png;base64,{b64}' "
            f"style='width:74px;height:74px;border:1px solid {hz.HZ_BORDER};"
            f"image-rendering:pixelated;background:#000;border-radius:4px'/>"
            f"<div style='font-size:11px;color:{hz.HZ_BLUE_DARK};font-weight:700'>{lbl}</div>"
            f"</div>")
    export_thumbs.value = (
        f"<div style='font-size:13px;font-weight:700;color:{hz.HZ_BLUE_DARK};margin:4px 0'>"
        f"Cropped ROIs ({len(cells)}):</div>" + "".join(cells))


# ── assignment + manifest ───────────────────────────────────────────────────────
def _render_bucket(*_):
    from IPython.display import HTML, display
    bucket = state["bucket"]
    bucket_out.clear_output(wait=True)
    with bucket_out:
        if not bucket:
            display(HTML("<i style='color:#5B6B7B;font-family:Arial'>Bucket is empty &mdash; load a "
                         "group JV file, then press <b>Assign (add bundle)</b>.</i>"))
            return
        bundles = {}
        for lbl, rec in bucket.items():
            bundles.setdefault(rec["jv_source_file"], []).append(lbl)
        njv = len(bucket); nbund = len(bundles); sealed = state["bucket_sealed"]
        head = (f"<div style='font-family:Arial'>"
                f"<b style='color:{hz.HZ_BLUE_DARK}'>Bucket &mdash; {nbund} bundle(s), {njv} ROI&harr;JV pair(s)</b>"
                f" <span style='color:{hz.HZ_MUTED};font-size:12px'>(best case 4 bundles / 24 pairs)</span>")
        tbl = ["<table style='border-collapse:collapse;font-size:13px;margin-top:6px'>",
               f"<tr style='color:{hz.HZ_BLUE_DARK}'>"
               f"<th style='padding:3px 12px;text-align:left'>Bundle (group JV file)</th>"
               f"<th style='padding:3px 16px'>ROI</th>"
               f"<th style='padding:3px 16px'>JV measurement</th>"
               f"<th style='padding:3px 12px'>PCE</th></tr>"]
        for src in sorted(bundles):
            members = sorted(bundles[src])
            for i, lbl in enumerate(members):
                rec = bucket[lbl]
                pce = rec["metrics"].get("PCE_rev", rec["metrics"].get("PCE_for"))
                pcecell = f"{pce:.2f}" if isinstance(pce, (int, float)) else ""
                srccell = (f"<span style='color:{hz.HZ_MUTED}'>{src}</span> "
                           f"<span style='color:{hz.HZ_BLUE}'>[{len(members)}]</span>") if i == 0 else ""
                tbl.append(
                    f"<tr><td style='padding:2px 12px'>{srccell}</td>"
                    f"<td style='padding:2px 16px;text-align:center;font-weight:700;"
                    f"background:{hz.HZ_TINT};color:{hz.HZ_BLUE_DARK};border-radius:4px'>{lbl}</td>"
                    f"<td style='padding:2px 16px;text-align:center;font-weight:700;color:{hz.HZ_BLUE}'>"
                    f"&#8594; {rec['device_id']}</td>"
                    f"<td style='padding:2px 12px;text-align:right'>{pcecell}</td></tr>")
        tbl.append("</table>")
        foot = (f"<div style='margin-top:8px;font-weight:700;"
                f"color:{hz.HZ_GREEN if sealed else hz.HZ_AMBER}'>"
                + ("&#10004; Bucket sealed &mdash; ready to export."
                   if sealed else "Bucket open &mdash; press <b>Seal the bucket</b> to enable export.")
                + "</div></div>")
        display(HTML(head + "".join(tbl) + foot))

def _reset_bucket_ui():
    state["bucket"] = {}; state["bucket_sealed"] = False
    try:
        btn_export_nomad.disabled = True
        _render_bucket()
    except Exception:
        pass

def add_to_bucket(_=None):
    rois = state["exported_rois"]; per_dev = state["per_device_jv"]
    with assign_out:
        from IPython.display import clear_output
        clear_output()
        if not rois:
            print("Export ROIs first (Step 3: Export ROI)."); return
        if not per_dev:
            print("Load a group JV file first."); return
        added, skipped = [], []
        for did, info in per_dev.items():
            if did in rois:
                state["bucket"][did] = {
                    "roi_label": did, "device_id": did, "quadrant": did.split("-")[0],
                    "jv_source_file": info["source_file"], "jv_bytes": info["bytes"],
                    "metrics": state["jv_metrics"].get(did, {}),
                    "roi_tiff": rois[did]["tiff"], "roi_record": rois[did]["record"],
                }
                added.append(did)
            else:
                skipped.append(did)
        state["bucket_sealed"] = False
        btn_export_nomad.disabled = True
        print(f"Added {len(added)} pair(s) to the bucket: {', '.join(sorted(added)) or '-'}")
        if skipped:
            print(f"No matching ROI for: {', '.join(sorted(skipped))} "
                  f"(crop those ROIs, or check quadrant<->C-N numbering).")
        print(f"Bucket now holds {len(state['bucket'])} pair(s). Load the next quadrant's JV "
              f"and Assign again, or Seal the bucket.")
    _render_bucket()

def seal_bucket(_=None):
    with assign_out:
        from IPython.display import clear_output
        clear_output()
        if not state["bucket"]:
            print("Bucket is empty - add at least one bundle first."); return
        state["bucket_sealed"] = True
        btn_export_nomad.disabled = False
        nb = len({r["jv_source_file"] for r in state["bucket"].values()})
        print(f"Bucket sealed: {nb} bundle(s), {len(state['bucket'])} pair(s). Export is now enabled.")
    _render_bucket()

btn_assign.on_click(add_to_bucket)
btn_seal.on_click(seal_bucket)


# ── bucket manifest + products ──────────────────────────────────────────────────
def _safe(name):
    return "".join(c if (c.isalnum() or c in "._-") else "_" for c in str(name or ""))

def _build_bucket_manifest(product_names):
    bucket = state["bucket"]; img = state["selected_image"] or {}
    bundles = {}
    for lbl, rec in bucket.items():
        bundles.setdefault(rec["jv_source_file"], []).append(lbl)
    bundle_list = []
    for src in sorted(bundles):
        pairs = []
        for lbl in sorted(bundles[src]):
            rec = bucket[lbl]
            pairs.append({
                "roi_label": lbl, "quadrant": rec["quadrant"], "device_id": rec["device_id"],
                "roi_tiff": product_names["roi"].get(lbl), "jv_file": product_names["jv"].get(lbl),
                "metrics": rec["metrics"], "geometry": rec["roi_record"],
            })
        bundle_list.append({"source_file": src, "n_pairs": len(pairs), "pairs": pairs})
    return {
        "format": "roi-jv-bucket", "version": 2,
        "created": _dt.datetime.now().isoformat(timespec="seconds"),
        "batch": state["batch_id"],
        "mother_image": {
            "filename": img.get("name"), "sample_id": img.get("sample_id"),
            "upload_id": img.get("upload_id"), "raw_path": img.get("path"),
            "overlay_png": product_names.get("overlay"),
        },
        "mask": os.path.basename(BUNDLED_DXF_PATH),
        "bucket": {"sealed": state["bucket_sealed"], "n_bundles": len(bundle_list),
                   "n_jv": len(bucket), "bundles": bundle_list},
    }

def _collect_products():
    """Build the sealed-bucket product set. Returns (files, names, folder_stamp, manifest, manifest_name)."""
    now = _dt.datetime.now()
    stamp = now.strftime("%d%m%Y-%H-%M-%S")           # DDMMYYYY-HH-MM-SS
    folder_stamp = now.strftime("%Y-%m-%d_%H%M%S")
    mother_stem = _safe(state["tiff_filename"] or "image")
    batch = _safe(state["batch_id"] or "batch")
    bucket = state["bucket"]
    files = {}; names = {"roi": {}, "jv": {}}

    overlay_name = "overlay.png"
    if state["overlay_png"]:
        files[overlay_name] = state["overlay_png"]
    names["overlay"] = overlay_name

    for lbl, rec in bucket.items():
        rn = f"ROI_{lbl}.tif"
        files[rn] = rec["roi_tiff"]; names["roi"][lbl] = rn
        jn = f"JV_{rec['device_id']}.txt"
        files[jn] = rec["jv_bytes"]; names["jv"][lbl] = jn

    manifest = _build_bucket_manifest(names)
    n_jv = len(bucket)
    manifest_name = f"ML-manifest-{batch}-{mother_stem}-{n_jv}-{stamp}.json"
    files[manifest_name] = json.dumps(manifest, indent=2).encode("utf-8")
    names["manifest"] = manifest_name
    return files, names, folder_stamp, manifest, manifest_name

def export_to_nomad(_=None):
    with nomad_out:
        from IPython.display import clear_output
        clear_output()
        if not state["bucket"]:
            print("Bucket is empty."); return
        if not state["bucket_sealed"]:
            print("Seal the bucket first."); return
        img = state["selected_image"]
        if not img:
            print("No mother image loaded."); return
        files, names, folder_stamp, manifest, manifest_name = _collect_products()
        subfolder = f"ML_bucket_{folder_stamp}"

        zbuf = io.BytesIO()
        with zipfile.ZipFile(zbuf, "w", zipfile.ZIP_DEFLATED) as z:
            for fn, data in files.items():
                z.writestr(f"{subfolder}/{fn}", data)
        b64 = base64.b64encode(zbuf.getvalue()).decode("ascii")
        manifest_dl.value = (f'<a download="{subfolder}.zip" '
                             f'href="data:application/zip;base64,{b64}">'
                             f'\u2b07 Download {subfolder}.zip (local copy)</a>')

        print(f"Bucket: {manifest['bucket']['n_bundles']} bundle(s), "
              f"{manifest['bucket']['n_jv']} pair(s).")
        print(f"Prepared {len(files)} products under {subfolder}/ :")
        print(f"  - 1 overlay PNG   - {len(names['roi'])} ROI TIFFs   "
              f"- {len(names['jv'])} JV txt   - {manifest_name}")
        try:
            nd.upload_products(state["nomad_url"], state["nomad_token"],
                               img["upload_id"], files, subfolder, out_widget=nomad_out)
            print(f"Uploaded to NOMAD upload {img['upload_id']} / {subfolder}/")
        except Exception as e:
            print(f"NOMAD upload failed ({e}).")
            print("The local .zip above still contains every product + the ML-manifest.")

btn_export_nomad.on_click(export_to_nomad)


# ── initial draw (now also auto-loads the bundled mask) ─────────────────────────
load_mask()
draw_image_preview(); draw_mask_preview(); draw_overlay_preview()
_refresh_export_lists()
_render_jv_chips()
_render_bucket()

# Per-user authentication: on NOMAD NORTH each session already carries THAT user's
# own access token in NOMAD_CLIENT_ACCESS_TOKEN. If it's present we connect right
# away, so a signed-in user just opens the app — no shared credentials, no token
# handling, and each person only sees the data their account is allowed to see.
if state["nomad_token"]:
    refresh_batches()

# Refresh button: force a full redraw of every preview canvas (clears the
# occasional torn/ghosted ipympl frame after loading a new or angled image).
btn_refresh_canvas = widgets.Button(description="↻ Refresh canvas",
    tooltip="Force a full redraw of the preview canvases")
def refresh_canvases(_=None):
    draw_image_preview(); draw_mask_preview(); draw_overlay_preview()
    for _f in (fig_img, fig_mag, fig_mask, fig_overlay, fig_omag):
        try: _f.canvas.draw(); _f.canvas.flush_events()
        except Exception: pass
    set_status("Canvas refreshed.", "#005AA0")
btn_refresh_canvas.on_click(refresh_canvases)


# ══════════════════════════════════════════════════════════════════════════════
# ── LAYOUT (Helmholtz theme) ──────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

# Auto-corner section (unchanged controls, themed container)
auto_corner_box = widgets.VBox([
    widgets.HTML("<b style='color:#1565C0'>🔍 Corner Snap</b>"),
    widgets.HBox([toggle_snap, radio_scope, radio_sensitivity]),
    widgets.HBox([widgets.HTML("<b style='color:#388E3C'>🔎 Overlay zoom:</b>&nbsp;"), radio_overlay_zoom]),
    corner_info,
], layout=widgets.Layout(border=f"1px solid {hz.HZ_BLUE_LIGHT}", padding="8px",
                         margin="4px 0", border_radius="6px"))

# Step 1 — pick source from NOMAD
source_card = hz.card([
    hz.step_label(1, "Select batch & image from NOMAD"),
    widgets.HBox([dd_batch, btn_refresh_batches, btn_load_batch, chk_only_tiff]),
    widgets.HBox([dd_image, btn_load_image]),
    source_out,
    hz.hint("Authenticated as your signed-in NOMAD session (per-user token) - no shared "
            "credentials. Type to filter batches; 'Load batch' lists its images. The 5x5 cell mask is bundled."),
])

# Step 2 — overlay / ROI controls (existing engine, unchanged behaviour)
overlay_card = hz.card([
    hz.step_label(2, "Overlay mask & tweak ROIs"),
    radio_mode,
    auto_corner_box,
    widgets.HBox([btn_load_mask, btn_reset_corners, btn_overlay, btn_export]),
    widgets.HBox([btn_reset_rois]),
    status, export_out,
])

controls = widgets.VBox([source_card, overlay_card])

row0 = widgets.HBox([fig_img.canvas,     fig_mag.canvas])     # image preview + corner magnifier
row1 = widgets.HBox([fig_overlay.canvas, fig_omag.canvas])    # overlay + overlay magnifier
row2 = widgets.HBox([fig_mask.canvas])                        # mask overview alone

# Step 3 — exported ROI thumbnails
thumbs_card = hz.card([
    hz.step_label(3, "Exported ROIs"),
    export_thumbs,
])

# Step 4 — JV load + bucket assignment
jv_panel_style = widgets.HTML(
    "<style>"
    ".jv-chipbox{resize:both;overflow:auto;border:1px solid #D5DEE7;border-radius:8px;"
    "padding:8px;background:#fff;min-width:280px;min-height:130px;max-width:100%;align-content:flex-start;}"
    ".jv-detail{border:1px solid #D5DEE7;border-radius:8px;padding:8px;background:#fff;"
    "min-width:300px;margin-left:8px;}"
    ".jvchip-btn{font-weight:700 !important;}"
    "</style>")
jv_chips_box.add_class("jv-chipbox")
jv_detail_panel = widgets.VBox([
    widgets.HTML("<b style='color:#002F5E;font-family:Arial'>JV viewer</b> "
                 "<span style='color:#5B6B7B;font-size:12px'>(click a chip)</span>"),
    jv_detail_out,
])
jv_detail_panel.add_class("jv-detail")

assign_card = hz.card([
    hz.step_label(4, "Assign ROIs to per-device JVs — bucket of bundles"),
    jv_panel_style,
    widgets.HBox([btn_browse_jv, jv_status]),
    widgets.HBox([sel_jv_browse, btn_load_selected_jv]),
    widgets.HTML("<b style='color:#002F5E;font-family:Arial'>Per-device JVs &mdash; "
                 "click a chip to view its curve &amp; metrics on the right "
                 "(drag the box corner to expand):</b>"),
    widgets.HBox([jv_chips_box, jv_detail_panel]),
    widgets.HBox([btn_assign, btn_seal, btn_export_nomad]),
    bucket_out,
    manifest_dl,
    assign_out,
    nomad_out,
])

app_root = widgets.VBox([
    hz.header("PL ROI -> JV Assignment", "NOMAD NORTH - Helmholtz-Zentrum Berlin"),
    controls,
    widgets.HBox([btn_refresh_canvas]),
    row0, row1, row2,
    thumbs_card, assign_card,
])
app_root.add_class("hz-app")
display(hz.global_css())
display(app_root)